# A trait-derived single-tile ecosystem

This notebook simulates isolated food webs whose species are authored entirely as sets of traits. A deterministic compiler turns those traits into numerical population parameters and diet relationships before the fast numerical solver runs.

Populations are total biomass within one land area, not individual counts. Size-class traits provide individual biomass, making approximate individual counts available when needed. There is no migration or ecosystem-specific calibration. Each tile receives the twelve monthly temperature, insolation, and precipitation samples produced by the Planet climate simulator.

In [ ]:
%use kandy

In [ ]:
import kotlin.math.PI
import kotlin.math.abs
import kotlin.math.log10
import kotlin.math.pow
import kotlin.math.roundToInt
import kotlin.math.sin
import kotlin.math.sqrt
import kotlin.random.Random
import dev.biserman.planet.planet.climate.ClimateDatum
import dev.biserman.planet.planet.climate.ClimateDatumSample
import dev.biserman.planet.planet.climate.ClimateClassification
import org.jetbrains.kotlinx.kandy.dsl.plot
import org.jetbrains.kotlinx.kandy.letsplot.layers.line

## Traits own their effects

A `SpeciesBlueprint` contains a stable id, a taxonomic order, and a set of traits. Each trait declares its own effects beside its enum constant. Numeric effects compose as summed offsets, multiplied factors, and stacked upper bounds; there is no override priority system. Other effects describe food resources, defenses, capabilities, requirements, and mutually exclusive groups. The fixed catalog has an exhaustive Order-to-Species mapping, while procedurally generated blueprints can provide their order directly.

Beneficial adaptations can also declare `MaintenanceCost` beside their benefits. These costs add together into an always-paid biomass loss, so a broadly overbuilt organism must continuously earn enough energy to maintain every adaptation. Range shifts and other mechanistic disadvantages remain separate trade-offs.

The compiler is intentionally generic. It folds numerical effects, matches `FeedsOn` with `EdibleAs`, applies defenses, and validates requirements without knowing that a particular trait is warm-blooded, fruiting, flying, or a pack hunter. Only universal relational mechanics, such as comparing two size classes, remain in the compiler.

At simulation assembly time, sessile autotrophs are grouped by size class and share one carrying-capacity density. Plant architecture exposes low foliage, canopy foliage, coarse grass, fruit, and seeds as separate resources; ground foraging, high browsing, and fermenting stomachs specialize how effectively a consumer exploits each layer beyond the limited access supplied by its basic grazer or leaf-eater niche. Fruit and seed traits define finite annual production rates shared by every consumer of that exact resource. Feeding species receive a resource-level overlap matrix, so eating berry foliage does not compete with eating berry fruit. Overlap increases feeding interference, while density-dependent mortality represents only self-crowding.

In [ ]:
/** Total biomass keyed by species id. */
typealias Biomass = Map<String, Double>

/** Marker interface for the declarative effects attached to a species trait. */
sealed interface TraitEffect

/** Capability and behavior labels used by requirements, conflicts, and counters. */
enum class EffectTag {
    AUTOTROPHY,      // Can gain biomass without eating another species.
    FEEDING,         // Can gain biomass through a derived diet.
    MOTILE,          // Has a mobile body and is available as motile prey.
    SESSILE,         // Is fixed in place.
    PACK_HUNTING,    // Hunts cooperatively and can tackle relatively larger prey.
    AMBUSH_HUNTING,  // Uses ambush tactics and partly counters some defenses.
    CHASE_HUNTING,   // Uses sustained pursuit.
    FLIGHT,          // Can fly and fully counters flight-based evasion.
    AERIAL_PREY_CAPTURE, // Can reliably seize small flying prey without flying itself.
    BURROWING,       // Can burrow and fully counters burrow-based evasion.
    NOCTURNAL,       // Is active at night and counters nocturnal evasion.
    WINTER_SURVIVAL, // Protects perennial plant tissue through freezing seasons.
}

/** Trait families in which a blueprint may select at most one member. */
enum class ExclusiveGroup {
    GROWTH_SPEED,     // Fast-growing and slow-growing are alternatives.
    HUNTING_STRATEGY, // Pack, ambush, and chase strategies are alternatives.
    COAT_DENSITY,     // Thick fur and a sparse coat are mutually exclusive.
    CAMOUFLAGE,       // Only one fixed or adaptive camouflage strategy may be selected.
}

/** Numerical solver parameters that trait effects may override or multiply. */
enum class NumericTarget {
    AUTOTROPHY_GROWTH_RATE,   // Intrinsic logistic growth rate of autotrophic biomass.
    CARRYING_CAPACITY,        // Baseline autotrophic biomass capacity per unit land area.
    SEASONAL_AMPLITUDE,       // Fractional seasonal oscillation in carrying capacity.
    MAX_CONSUMPTION_RATE,     // Maximum food biomass ingested per feeder biomass per year.
    ASSIMILATION_EFFICIENCY,  // Fraction of ingested biomass converted into feeder biomass.
    MORTALITY_RATE,           // Baseline feeder biomass lost per unit biomass per year.
    METABOLIC_DEMAND_RATE,    // Assimilated food needed per feeder biomass per year to avoid starvation.
    STARVATION_MORTALITY_RATE, // Maximum fractional biomass loss per year at zero food intake.
    DENSITY_DEPENDENCE,       // Strength of crowding, disease, and territorial losses.
    INTERFERENCE,             // Strength with which crowded feeders obstruct feeding.
    RESPONSE_EXPONENT,        // Exponent of the feeding response; values above one protect scarce prey.
    MIN_OPTIMUM_TEMPERATURE_C, // Lower edge of the no-stress temperature interval in degrees Celsius.
    MAX_OPTIMUM_TEMPERATURE_C, // Upper edge of the no-stress temperature interval in degrees Celsius.
    MIN_OPTIMUM_MOISTURE_MM,   // Lowest no-stress monthly precipitation in millimetres.
    MAX_OPTIMUM_MOISTURE_MM,   // Highest no-stress monthly precipitation; infinity means no upper limit.
    MIN_OPTIMUM_INSOLATION_W_M2, // Lowest no-stress mean insolation in watts per square metre.
    MAX_OPTIMUM_INSOLATION_W_M2, // Highest no-stress mean insolation in watts per square metre.
    PHOTOSYNTHESIS_HALF_SATURATION_W_M2, // Insolation producing half of the maximum light response.
    CLIMATE_STRESS_SENSITIVITY, // Multiplier applied to mortality caused by climate outside the niche.
}

/** Abstract resource categories used to match feeding niches with edible features. */
enum class FoodResource {
    LOW_FOLIAGE,    // Leaves and shoots reachable by ground-level foragers.
    CANOPY_FOLIAGE, // Elevated foliage accessible to high browsers.
    COARSE_GRASS,   // Fibrous grass best exploited by fermenting grazers.
    BAMBOO,         // Bamboo leaves, shoots, and culms targeted by bamboo specialists.
    FRUIT,        // Renewable fruit offered without standing-biomass loss.
    SEED,         // Seeds, with only a small standing-biomass cost.
    WOOD,         // Woody tissue.
    MOTILE_PREY,  // The body biomass of a motile organism.
}

/** Adds a capability or countermeasure label to a blueprint. */
data class GrantsTag(
    /** Tag made available to validation and interaction effects. */
    val tag: EffectTag,
) : TraitEffect

/** Declares that another effect on the same blueprint must grant a tag. */
data class RequiresTag(
    /** Tag that must be present after all traits are combined. */
    val tag: EffectTag,
) : TraitEffect

/** Declares that another effect on the same blueprint must not grant a tag. */
data class ConflictsWithTag(
    /** Tag whose presence makes the blueprint invalid. */
    val tag: EffectTag,
) : TraitEffect

/** Places a trait into a mutually exclusive family. */
data class ExclusiveEffect(
    /** Family in which only one effect may occur on a blueprint. */
    val group: ExclusiveGroup,
) : TraitEffect

/** Composably caps a numerical parameter after offsets and multipliers are applied. */
data class NumericUpperBound(
    /** Solver parameter being capped. */
    val target: NumericTarget,
    /** Maximum allowed value; multiple bounds stack by choosing the smallest. */
    val maximum: Double,
) : TraitEffect

/** Multiplies a numerical parameter after all additive effects are summed. */
data class NumericMultiplier(
    /** Solver parameter being scaled. */
    val target: NumericTarget,
    /** Factor multiplied into the selected parameter value. */
    val multiplier: Double,
) : TraitEffect

/** Adds a signed amount to a numerical parameter before multipliers are applied. */
data class NumericOffset(
    /** Solver parameter being shifted. */
    val target: NumericTarget,
    /** Signed shift in the target parameter's own units. */
    val offset: Double,
) : TraitEffect

/** Adds an always-paid energetic and structural biomass cost to a trait. */
data class MaintenanceCost(
    /** Fractional biomass lost per year while maintaining this adaptation. */
    val biomassLossRatePerYear: Double,
) : TraitEffect {
    init {
        require(biomassLossRatePerYear >= 0.0)
    }
}

/** Exposes part of a species as a resource that feeding traits can target. */
data class EdibleAs(
    /** Resource category exposed by this trait. */
    val resource: FoodResource,
    /** Multiplier describing how available or attractive the resource is. */
    val preferenceMultiplier: Double = 1.0,
    /** Fraction of ingested biomass removed from the prey's standing biomass. */
    val preyLossFraction: Double = 1.0,
    /** Maximum renewable food produced per unit producer biomass per year; null means standing biomass. */
    val renewableProductionRate: Double? = null,
) : TraitEffect {
    init {
        require(preferenceMultiplier > 0.0)
        require(preyLossFraction in 0.0..1.0)
        require(renewableProductionRate == null || renewableProductionRate > 0.0)
    }
}

/** Defines a feeding niche for one resource category. */
data class FeedsOn(
    /** Resource category sought by the feeder. */
    val resource: FoodResource,
    /** Base preference before prey features and defenses are applied. */
    val preference: Double,
    /** Optional absolute upper size rank for eligible prey. */
    val maximumPreySizeRank: Int? = null,
    /** Optional smallest prey-minus-feeder size-rank difference. */
    val minimumRelativeSize: Int? = null,
    /** Optional largest prey-minus-feeder size-rank difference. */
    val maximumRelativeSize: Int? = null,
) : TraitEffect

/** Expands a relative-size feeding niche, as cooperative hunting does. */
data class RelativeSizeModifier(
    /** Number of additional prey size ranks that can be attacked. */
    val maximumRelativeSizeBonus: Int = 0,
) : TraitEffect

/** Reduces preference for a defended prey unless the feeder has a counter. */
data class DefenseEffect(
    /** Multiplier used when the feeder has no applicable countermeasure. */
    val defaultPreferenceMultiplier: Double,
    /** Preference multipliers selected when the feeder grants a matching counter tag. */
    val counterMultipliers: Map<EffectTag, Double> = emptyMap(),
    /** Optional feeder size advantage at which this defense stops mattering. */
    val ignoredWhenFeederSizeAdvantageAtLeast: Int? = null,
) : TraitEffect

/** Habitat dimensions that traits can require or use as refuges. */
enum class HabitatAxis {
    CANOPY_CLOSURE,       // Fraction of overhead light intercepted by large canopies.
    WETLAND_AVAILABILITY, // Fractional wetland availability derived from current precipitation.
}

/** Makes a sessile producer intercept light before it reaches shorter plants. */
data class CanopyEffect(
    /** Fraction of incident light intercepted when this canopy fills its size guild. */
    val maximumLightInterception: Double,
) : TraitEffect {
    init { require(maximumLightInterception in 0.0..1.0) }
}

/** Adds mortality when a habitat axis lies outside a trait's usable interval. */
data class HabitatPreferenceEffect(
    val axis: HabitatAxis,
    val minimum: Double = 0.0,
    val maximum: Double = 1.0,
    val mortalityAtExtreme: Double,
) : TraitEffect {
    init {
        require(minimum in 0.0..1.0 && maximum in minimum..1.0)
        require(mortalityAtExtreme >= 0.0)
    }
}

/** Reduces predation while the prey can use a sufficiently available habitat refuge. */
data class HabitatRefugeEffect(
    val axis: HabitatAxis,
    val refugeStartsAt: Double,
    val predatorPreferenceAtFullRefuge: Double,
) : TraitEffect {
    init {
        require(refugeStartsAt in 0.0..<1.0)
        require(predatorPreferenceAtFullRefuge in 0.0..1.0)
    }
}

/** Extra cold injury caused by exposed tissue or poorly insulated extremities. */
data class ColdExposureEffect(
    val mortalityPerDegreeBelowFreezing: Double,
) : TraitEffect {
    init { require(mortalityPerDegreeBelowFreezing >= 0.0) }
}

/** Extra overheating caused by insulation that cannot be shed in hot conditions. */
data class HeatRetentionEffect(
    val mortalityPerDegreeAboveThreshold: Double,
    val thresholdC: Double = 20.0,
) : TraitEffect {
    init { require(mortalityPerDegreeAboveThreshold >= 0.0) }
}

/** Raises the elevation above which hypoxia adds mortality. */
data class AltitudeToleranceEffect(
    val maximumOptimalAltitudeMeters: Double,
    val mortalityPerKilometerAbove: Double = 0.45,
) : TraitEffect {
    init {
        require(maximumOptimalAltitudeMeters >= 0.0)
        require(mortalityPerKilometerAbove >= 0.0)
    }
}

/** Visual background against which a fixed camouflage pattern is effective. */
enum class CamouflageHabitat { SNOW, DESERT, GRASSLAND, FOREST, ADAPTIVE }

/** Modifies hunting success and vulnerability according to current background match. */
data class CamouflageEffect(
    val habitat: CamouflageHabitat,
    val predatorMultiplierWhenMatched: Double = 1.15,
    val preyMultiplierWhenMatched: Double = 0.55,
    val predatorMultiplierWhenMismatched: Double = 0.72,
    val preyMultiplierWhenMismatched: Double = 1.40,
) : TraitEffect

/** Always-visible mate signaling that also makes hunting and concealment harder. */
data class ConspicuousDisplayEffect(
    val predatorHuntingMultiplier: Double,
    val preyVulnerabilityMultiplier: Double,
) : TraitEffect

/** A composable item that may be placed in a species blueprint. */
sealed interface SpeciesTrait {
    /** Declarative effects contributed when this trait appears in a blueprint. */
    val effects: List<TraitEffect>
}

/** Body-size trait, including the biomass represented by one individual. */
enum class SizeClass(
    /** Ordered size rank used by predator-prey compatibility rules. */
    val rank: Int,
    /** Biomass of one representative individual in simulation biomass units. */
    val individualBiomass: Double,
    /** Default shared carrying-capacity density for sessile producers of this size. */
    val defaultSessileCarryingCapacityDensity: Double,
    /** Temperature-niche shift predicted by Bergmann's law for motile organisms. */
    val bergmannTemperatureShiftC: Double,
) : SpeciesTrait {
    MINUSCULE(0, 0.0001, 1.00, 4.0),
    TINY(1, 0.01, 0.80, 2.5),
    SMALL(2, 1.0, 1.10, 1.0),
    MEDIUM(3, 20.0, 1.40, 0.0),
    LARGE(4, 200.0, 1.70, -1.5),
    GIANT(5, 2_000.0, 2.00, -3.0),
    COLOSSAL(6, 20_000.0, 2.40, -4.5);

    /** Size classes contribute their numbers directly rather than descriptor effects. */
    override val effects: List<TraitEffect> = emptyList()

    /** Allometric multiplier used by baseline feeding and mortality rates. */
    val feedingAllometry: Double = individualBiomass.pow(-0.08)
}

/** Named ecological traits; every constant declares its complete effect bundle here. */
enum class Trait(
    /** Effects contributed by the enum constant. */
    vararg effectList: TraitEffect,
) : SpeciesTrait {
    SESSILE_PRODUCER(
        GrantsTag(EffectTag.AUTOTROPHY),
        GrantsTag(EffectTag.SESSILE),
        ConflictsWithTag(EffectTag.MOTILE),
    ),
    PHOTOSYNTHETIC(
        GrantsTag(EffectTag.AUTOTROPHY),
        MaintenanceCost(0.015),
        NumericMultiplier(NumericTarget.AUTOTROPHY_GROWTH_RATE, 0.30),
        NumericMultiplier(NumericTarget.CARRYING_CAPACITY, 0.35),
    ),
    GRASS_LIKE(
        MaintenanceCost(0.002),
        NumericOffset(NumericTarget.CARRYING_CAPACITY, 0.45),
    ),
    LEAFY(
        MaintenanceCost(0.003),
        NumericOffset(NumericTarget.CARRYING_CAPACITY, 0.25),
    ),
    LOW_FOLIAGE(
        RequiresTag(EffectTag.AUTOTROPHY),
        EdibleAs(FoodResource.LOW_FOLIAGE),
    ),
    CANOPY_FOLIAGE(
        RequiresTag(EffectTag.AUTOTROPHY),
        EdibleAs(FoodResource.CANOPY_FOLIAGE),
        MaintenanceCost(0.003),
    ),
    LARGE_CANOPY(
        RequiresTag(EffectTag.SESSILE),
        CanopyEffect(maximumLightInterception = 0.95),
        MaintenanceCost(0.012),
        NumericMultiplier(NumericTarget.AUTOTROPHY_GROWTH_RATE, 0.92),
    ),
    COARSE_GRASS(
        RequiresTag(EffectTag.AUTOTROPHY),
        EdibleAs(FoodResource.COARSE_GRASS, preferenceMultiplier = 0.85),
        MaintenanceCost(0.002),
        NumericMultiplier(NumericTarget.AUTOTROPHY_GROWTH_RATE, 0.95),
    ),
    BAMBOO_CULMS(
        RequiresTag(EffectTag.AUTOTROPHY),
        EdibleAs(FoodResource.BAMBOO, preferenceMultiplier = 1.0),
        MaintenanceCost(0.004),
        NumericOffset(NumericTarget.CARRYING_CAPACITY, 0.30),
        NumericMultiplier(NumericTarget.MIN_OPTIMUM_MOISTURE_MM, 1.75),
        NumericOffset(NumericTarget.MIN_OPTIMUM_TEMPERATURE_C, -4.0),
        NumericOffset(NumericTarget.MAX_OPTIMUM_TEMPERATURE_C, 4.0),
    ),
    RHIZOMATOUS_GROWTH(
        RequiresTag(EffectTag.AUTOTROPHY),
        GrantsTag(EffectTag.WINTER_SURVIVAL),
        MaintenanceCost(0.006),
        NumericMultiplier(NumericTarget.AUTOTROPHY_GROWTH_RATE, 1.15),
        NumericOffset(NumericTarget.MIN_OPTIMUM_TEMPERATURE_C, -8.0),
        NumericOffset(NumericTarget.MAX_OPTIMUM_TEMPERATURE_C, 2.0),
    ),
    DRIP_TIP_LEAVES(
        RequiresTag(EffectTag.AUTOTROPHY),
        MaintenanceCost(0.004),
        NumericMultiplier(NumericTarget.MIN_OPTIMUM_MOISTURE_MM, 1.50),
        NumericOffset(NumericTarget.MIN_OPTIMUM_TEMPERATURE_C, 4.0),
        NumericOffset(NumericTarget.MAX_OPTIMUM_TEMPERATURE_C, 4.0),
        NumericMultiplier(NumericTarget.CLIMATE_STRESS_SENSITIVITY, 0.85),
    ),
    FRUITING(
        EdibleAs(
            FoodResource.FRUIT,
            preyLossFraction = 0.1,
            renewableProductionRate = 0.30,
        ),
        MaintenanceCost(0.015),
    ),
    SEED_BEARING(
        EdibleAs(
            FoodResource.SEED,
            preyLossFraction = 0.05,
            renewableProductionRate = 0.08,
        ),
        MaintenanceCost(0.008),
    ),
    WOODY(
        EdibleAs(FoodResource.WOOD),
        NumericOffset(NumericTarget.CARRYING_CAPACITY, 1.05),
        NumericMultiplier(NumericTarget.AUTOTROPHY_GROWTH_RATE, 0.50),
    ),
    FAST_GROWING(
        NumericMultiplier(NumericTarget.AUTOTROPHY_GROWTH_RATE, 1.70),
        MaintenanceCost(0.020),
        ExclusiveEffect(ExclusiveGroup.GROWTH_SPEED),
    ),
    SLOW_GROWING(
        NumericMultiplier(NumericTarget.AUTOTROPHY_GROWTH_RATE, 0.32),
        ExclusiveEffect(ExclusiveGroup.GROWTH_SPEED),
    ),
    EVERGREEN(
        NumericMultiplier(NumericTarget.SEASONAL_AMPLITUDE, 0.28),
        NumericOffset(NumericTarget.MIN_OPTIMUM_TEMPERATURE_C, -5.0),
        MaintenanceCost(0.008),
    ),

    GRASS_GRAZER(
        GrantsTag(EffectTag.FEEDING),
        MaintenanceCost(0.004),
        FeedsOn(FoodResource.LOW_FOLIAGE, preference = 0.55),
        FeedsOn(FoodResource.COARSE_GRASS, preference = 0.35),
    ),
    BAMBOO_SPECIALIST(
        GrantsTag(EffectTag.FEEDING),
        MaintenanceCost(0.006),
        FeedsOn(FoodResource.BAMBOO, preference = 1.35),
        NumericMultiplier(NumericTarget.MAX_CONSUMPTION_RATE, 1.60),
        NumericMultiplier(NumericTarget.ASSIMILATION_EFFICIENCY, 0.65),
    ),
    LEAF_EATER(
        GrantsTag(EffectTag.FEEDING),
        MaintenanceCost(0.004),
        FeedsOn(FoodResource.LOW_FOLIAGE, preference = 0.55),
        FeedsOn(FoodResource.CANOPY_FOLIAGE, preference = 0.35),
    ),
    GROUND_FORAGING(
        RequiresTag(EffectTag.FEEDING),
        MaintenanceCost(0.004),
        FeedsOn(FoodResource.LOW_FOLIAGE, preference = 1.00),
    ),
    HIGH_BROWSING(
        RequiresTag(EffectTag.FEEDING),
        MaintenanceCost(0.008),
        FeedsOn(FoodResource.CANOPY_FOLIAGE, preference = 1.00),
    ),
    FRUGIVORE(
        GrantsTag(EffectTag.FEEDING),
        MaintenanceCost(0.006),
        FeedsOn(FoodResource.FRUIT, preference = 1.10),
    ),
    SEED_EATER(
        GrantsTag(EffectTag.FEEDING),
        MaintenanceCost(0.004),
        FeedsOn(FoodResource.SEED, preference = 0.75),
    ),
    DENDROVORE(
        GrantsTag(EffectTag.FEEDING),
        MaintenanceCost(0.006),
        FeedsOn(FoodResource.WOOD, preference = 0.70),
    ),
    INSECTIVORE(
        GrantsTag(EffectTag.FEEDING),
        MaintenanceCost(0.006),
        FeedsOn(
            FoodResource.MOTILE_PREY,
            preference = 0.95,
            maximumPreySizeRank = SizeClass.TINY.rank,
        ),
    ),
    FILTER_FEEDER(
        GrantsTag(EffectTag.FEEDING),
        MaintenanceCost(0.008),
        FeedsOn(
            FoodResource.MOTILE_PREY,
            preference = 1.00,
            maximumPreySizeRank = SizeClass.TINY.rank,
        ),
        NumericMultiplier(NumericTarget.MAX_CONSUMPTION_RATE, 1.15),
    ),
    BENTHIC_FORAGER(
        GrantsTag(EffectTag.FEEDING),
        MaintenanceCost(0.006),
        FeedsOn(
            FoodResource.MOTILE_PREY,
            preference = 0.90,
            maximumPreySizeRank = SizeClass.SMALL.rank,
        ),
    ),
    CARNIVORE(
        GrantsTag(EffectTag.FEEDING),
        MaintenanceCost(0.008),
        FeedsOn(
            FoodResource.MOTILE_PREY,
            preference = 1.00,
            minimumRelativeSize = -2,
            maximumRelativeSize = 0,
        ),
    ),
    OMNIVORE(
        GrantsTag(EffectTag.FEEDING),
        MaintenanceCost(0.012),
        FeedsOn(
            FoodResource.MOTILE_PREY,
            preference = 0.60,
            minimumRelativeSize = -2,
            maximumRelativeSize = 0,
        ),
        FeedsOn(FoodResource.FRUIT, preference = 0.75),
        FeedsOn(FoodResource.SEED, preference = 0.55),
    ),

    MOTILE(
        GrantsTag(EffectTag.MOTILE),
        MaintenanceCost(0.010),
        EdibleAs(FoodResource.MOTILE_PREY),
    ),
    WARM_BLOODED(
        RequiresTag(EffectTag.MOTILE),
        NumericMultiplier(NumericTarget.MAX_CONSUMPTION_RATE, 5.0),
        NumericMultiplier(NumericTarget.METABOLIC_DEMAND_RATE, 5.0),
        NumericMultiplier(NumericTarget.STARVATION_MORTALITY_RATE, 1.50),
        NumericMultiplier(NumericTarget.MORTALITY_RATE, 1.15),
        NumericOffset(NumericTarget.MIN_OPTIMUM_TEMPERATURE_C, -8.0),
    ),
    PACK_HUNTER(
        GrantsTag(EffectTag.PACK_HUNTING),
        RequiresTag(EffectTag.FEEDING),
        MaintenanceCost(0.012),
        ExclusiveEffect(ExclusiveGroup.HUNTING_STRATEGY),
        NumericMultiplier(NumericTarget.MAX_CONSUMPTION_RATE, 1.05),
        NumericMultiplier(NumericTarget.DENSITY_DEPENDENCE, 0.80),
        NumericMultiplier(NumericTarget.INTERFERENCE, 0.58),
        RelativeSizeModifier(maximumRelativeSizeBonus = 1),
    ),
    AMBUSH_PREDATOR(
        GrantsTag(EffectTag.AMBUSH_HUNTING),
        RequiresTag(EffectTag.FEEDING),
        MaintenanceCost(0.005),
        ExclusiveEffect(ExclusiveGroup.HUNTING_STRATEGY),
        NumericMultiplier(NumericTarget.MAX_CONSUMPTION_RATE, 0.90),
        NumericMultiplier(NumericTarget.INTERFERENCE, 0.75),
        NumericMultiplier(NumericTarget.RESPONSE_EXPONENT, 1.20),
    ),
    CHASE_PREDATOR(
        GrantsTag(EffectTag.CHASE_HUNTING),
        RequiresTag(EffectTag.FEEDING),
        MaintenanceCost(0.010),
        ExclusiveEffect(ExclusiveGroup.HUNTING_STRATEGY),
        NumericMultiplier(NumericTarget.MAX_CONSUMPTION_RATE, 1.15),
        NumericMultiplier(NumericTarget.INTERFERENCE, 1.25),
    ),
    FLIGHT(
        GrantsTag(EffectTag.FLIGHT),
        RequiresTag(EffectTag.MOTILE),
        MaintenanceCost(0.045),
        DefenseEffect(
            defaultPreferenceMultiplier = 0.08,
            counterMultipliers = mapOf(
                EffectTag.FLIGHT to 1.0,
                EffectTag.AMBUSH_HUNTING to 0.30,
                EffectTag.AERIAL_PREY_CAPTURE to 0.75,
            ),
        ),
    ),
    BURROWING(
        GrantsTag(EffectTag.BURROWING),
        RequiresTag(EffectTag.MOTILE),
        MaintenanceCost(0.020),
        NumericOffset(NumericTarget.MIN_OPTIMUM_TEMPERATURE_C, -2.0),
        NumericOffset(NumericTarget.MAX_OPTIMUM_TEMPERATURE_C, 2.0),
        NumericUpperBound(NumericTarget.MAX_OPTIMUM_MOISTURE_MM, 140.0),
        DefenseEffect(
            defaultPreferenceMultiplier = 0.30,
            counterMultipliers = mapOf(
                EffectTag.BURROWING to 1.0,
                EffectTag.AMBUSH_HUNTING to 0.65,
            ),
        ),
    ),
    ARMORED(
        MaintenanceCost(0.025),
        DefenseEffect(
            defaultPreferenceMultiplier = 0.35,
            ignoredWhenFeederSizeAdvantageAtLeast = 2,
        ),
    ),
    VENOMOUS(
        MaintenanceCost(0.018),
        DefenseEffect(defaultPreferenceMultiplier = 0.75),
    ),
    NOCTURNAL(
        GrantsTag(EffectTag.NOCTURNAL),
        MaintenanceCost(0.005),
        NumericOffset(NumericTarget.MAX_OPTIMUM_TEMPERATURE_C, 3.0),
        NumericOffset(NumericTarget.MIN_OPTIMUM_TEMPERATURE_C, 3.0),
        NumericMultiplier(NumericTarget.MIN_OPTIMUM_INSOLATION_W_M2, 0.35),
        NumericMultiplier(NumericTarget.MAX_OPTIMUM_INSOLATION_W_M2, 0.70),
        DefenseEffect(
            defaultPreferenceMultiplier = 0.60,
            counterMultipliers = mapOf(EffectTag.NOCTURNAL to 1.0),
        ),
    ),
    REFLECTIVE_RETINA(
        RequiresTag(EffectTag.MOTILE),
        MaintenanceCost(0.006),
        NumericMultiplier(NumericTarget.MIN_OPTIMUM_INSOLATION_W_M2, 0.55),
        NumericMultiplier(NumericTarget.MAX_OPTIMUM_INSOLATION_W_M2, 0.80),
    ),
    LARGE_EYES(
        RequiresTag(EffectTag.MOTILE),
        MaintenanceCost(0.008),
        NumericMultiplier(NumericTarget.MIN_OPTIMUM_INSOLATION_W_M2, 0.70),
        NumericMultiplier(NumericTarget.MAX_OPTIMUM_INSOLATION_W_M2, 0.90),
    ),
    CONSTRICTING_PUPILS(
        RequiresTag(EffectTag.MOTILE),
        MaintenanceCost(0.005),
        NumericMultiplier(NumericTarget.MIN_OPTIMUM_INSOLATION_W_M2, 0.85),
        NumericMultiplier(NumericTarget.MAX_OPTIMUM_INSOLATION_W_M2, 1.40),
    ),
    UV_FILTERING_LENSES(
        RequiresTag(EffectTag.MOTILE),
        MaintenanceCost(0.004),
        NumericMultiplier(NumericTarget.MAX_OPTIMUM_INSOLATION_W_M2, 1.50),
    ),
    AQUATIC(
        MaintenanceCost(0.012),
        NumericMultiplier(NumericTarget.CARRYING_CAPACITY, 1.10),
        NumericMultiplier(NumericTarget.SEASONAL_AMPLITUDE, 0.67),
        NumericMultiplier(NumericTarget.MIN_OPTIMUM_MOISTURE_MM, 0.0),
    ),
    THICK_FUR(
        RequiresTag(EffectTag.MOTILE),
        ExclusiveEffect(ExclusiveGroup.COAT_DENSITY),
        MaintenanceCost(0.008),
        NumericOffset(NumericTarget.MIN_OPTIMUM_TEMPERATURE_C, -6.0),
        NumericOffset(NumericTarget.MAX_OPTIMUM_TEMPERATURE_C, -2.0),
        HeatRetentionEffect(mortalityPerDegreeAboveThreshold = 0.020),
    ),
    BLUBBER(
        RequiresTag(EffectTag.MOTILE),
        MaintenanceCost(0.015),
        NumericMultiplier(NumericTarget.STARVATION_MORTALITY_RATE, 0.75),
        NumericOffset(NumericTarget.MIN_OPTIMUM_TEMPERATURE_C, -8.0),
        NumericOffset(NumericTarget.MAX_OPTIMUM_TEMPERATURE_C, -4.0),
        HeatRetentionEffect(mortalityPerDegreeAboveThreshold = 0.030),
    ),
    SUBCUTANEOUS_FAT(
        RequiresTag(EffectTag.MOTILE),
        MaintenanceCost(0.014),
        NumericMultiplier(NumericTarget.STARVATION_MORTALITY_RATE, 0.80),
        NumericOffset(NumericTarget.MIN_OPTIMUM_TEMPERATURE_C, -6.0),
        NumericOffset(NumericTarget.MAX_OPTIMUM_TEMPERATURE_C, -7.0),
        HeatRetentionEffect(mortalityPerDegreeAboveThreshold = 0.035, thresholdC = 16.0),
    ),
    COUNTERCURRENT_HEAT_EXCHANGE(
        RequiresTag(EffectTag.MOTILE),
        MaintenanceCost(0.012),
        NumericOffset(NumericTarget.MIN_OPTIMUM_TEMPERATURE_C, -4.0),
    ),
    THICK_BARK(
        RequiresTag(EffectTag.AUTOTROPHY),
        MaintenanceCost(0.006),
        NumericOffset(NumericTarget.MIN_OPTIMUM_TEMPERATURE_C, -4.0),
    ),
    DORMANT_BUDS(
        RequiresTag(EffectTag.AUTOTROPHY),
        GrantsTag(EffectTag.WINTER_SURVIVAL),
        MaintenanceCost(0.006),
        NumericOffset(NumericTarget.MIN_OPTIMUM_TEMPERATURE_C, -6.0),
        NumericOffset(NumericTarget.MAX_OPTIMUM_TEMPERATURE_C, -2.0),
    ),
    UNDERGROUND_MERISTEMS(
        RequiresTag(EffectTag.AUTOTROPHY),
        GrantsTag(EffectTag.WINTER_SURVIVAL),
        MaintenanceCost(0.006),
        NumericOffset(NumericTarget.MIN_OPTIMUM_TEMPERATURE_C, -35.0),
        NumericOffset(NumericTarget.MAX_OPTIMUM_TEMPERATURE_C, -4.0),
        NumericMultiplier(NumericTarget.MIN_OPTIMUM_MOISTURE_MM, 0.55),
        NumericMultiplier(NumericTarget.CLIMATE_STRESS_SENSITIVITY, 0.35),
    ),
    NEEDLE_LEAVES(
        RequiresTag(EffectTag.AUTOTROPHY),
        GrantsTag(EffectTag.WINTER_SURVIVAL),
        MaintenanceCost(0.006),
        NumericMultiplier(NumericTarget.AUTOTROPHY_GROWTH_RATE, 0.80),
        NumericOffset(NumericTarget.MIN_OPTIMUM_TEMPERATURE_C, -8.0),
        NumericOffset(NumericTarget.MAX_OPTIMUM_TEMPERATURE_C, -2.0),
        NumericMultiplier(NumericTarget.MIN_OPTIMUM_MOISTURE_MM, 0.45),
        NumericMultiplier(NumericTarget.CLIMATE_STRESS_SENSITIVITY, 0.55),
    ),
    PROSTRATE_WOODY_GROWTH(
        RequiresTag(EffectTag.SESSILE),
        MaintenanceCost(0.004),
        NumericMultiplier(NumericTarget.AUTOTROPHY_GROWTH_RATE, 0.70),
        NumericMultiplier(NumericTarget.CARRYING_CAPACITY, 0.45),
        NumericOffset(NumericTarget.MIN_OPTIMUM_TEMPERATURE_C, -24.0),
        NumericOffset(NumericTarget.MAX_OPTIMUM_TEMPERATURE_C, -6.0),
        NumericMultiplier(NumericTarget.MIN_OPTIMUM_MOISTURE_MM, 0.55),
        NumericMultiplier(NumericTarget.CLIMATE_STRESS_SENSITIVITY, 0.45),
        HeatRetentionEffect(mortalityPerDegreeAboveThreshold = 0.060, thresholdC = 14.0),
    ),
    LARGE_HEAT_RADIATING_EARS(
        RequiresTag(EffectTag.MOTILE),
        MaintenanceCost(0.005),
        NumericOffset(NumericTarget.MIN_OPTIMUM_TEMPERATURE_C, 2.0),
        NumericOffset(NumericTarget.MAX_OPTIMUM_TEMPERATURE_C, 7.0),
        ColdExposureEffect(mortalityPerDegreeBelowFreezing = 0.025),
    ),
    SPARSE_COAT(
        RequiresTag(EffectTag.MOTILE),
        ExclusiveEffect(ExclusiveGroup.COAT_DENSITY),
        NumericOffset(NumericTarget.MIN_OPTIMUM_TEMPERATURE_C, 5.0),
        NumericOffset(NumericTarget.MAX_OPTIMUM_TEMPERATURE_C, 5.0),
        ColdExposureEffect(mortalityPerDegreeBelowFreezing = 0.050),
    ),
    ESTIVATION(
        RequiresTag(EffectTag.MOTILE),
        MaintenanceCost(0.012),
        NumericOffset(NumericTarget.MAX_OPTIMUM_TEMPERATURE_C, 6.0),
        NumericMultiplier(NumericTarget.MIN_OPTIMUM_MOISTURE_MM, 0.80),
    ),
    WAXY_CUTICLE(
        RequiresTag(EffectTag.AUTOTROPHY),
        MaintenanceCost(0.008),
        NumericMultiplier(NumericTarget.MIN_OPTIMUM_MOISTURE_MM, 0.70),
    ),
    DEEP_ROOTS(
        RequiresTag(EffectTag.SESSILE),
        MaintenanceCost(0.012),
        NumericMultiplier(NumericTarget.MIN_OPTIMUM_MOISTURE_MM, 0.60),
    ),
    WATER_STORAGE_TISSUE(
        RequiresTag(EffectTag.AUTOTROPHY),
        MaintenanceCost(0.015),
        NumericMultiplier(NumericTarget.MIN_OPTIMUM_MOISTURE_MM, 0.70),
    ),
    CAM_PHOTOSYNTHESIS(
        RequiresTag(EffectTag.AUTOTROPHY),
        MaintenanceCost(0.008),
        NumericMultiplier(NumericTarget.AUTOTROPHY_GROWTH_RATE, 0.55),
        NumericMultiplier(NumericTarget.MIN_OPTIMUM_MOISTURE_MM, 0.25),
        NumericMultiplier(NumericTarget.CLIMATE_STRESS_SENSITIVITY, 0.40),
        NumericUpperBound(NumericTarget.MAX_OPTIMUM_MOISTURE_MM, 80.0),
        NumericOffset(NumericTarget.MIN_OPTIMUM_TEMPERATURE_C, 5.0),
        NumericOffset(NumericTarget.MAX_OPTIMUM_TEMPERATURE_C, 10.0),
    ),
    DROUGHT_DECIDUOUS_FOLIAGE(
        RequiresTag(EffectTag.AUTOTROPHY),
        MaintenanceCost(0.004),
        NumericMultiplier(NumericTarget.AUTOTROPHY_GROWTH_RATE, 0.85),
        NumericMultiplier(NumericTarget.MIN_OPTIMUM_MOISTURE_MM, 0.30),
        NumericMultiplier(NumericTarget.CLIMATE_STRESS_SENSITIVITY, 0.35),
        NumericUpperBound(NumericTarget.MAX_OPTIMUM_MOISTURE_MM, 140.0),
        NumericOffset(NumericTarget.MAX_OPTIMUM_TEMPERATURE_C, 6.0),
    ),
    CONCENTRATED_URINE(
        RequiresTag(EffectTag.MOTILE),
        MaintenanceCost(0.010),
        NumericMultiplier(NumericTarget.MIN_OPTIMUM_MOISTURE_MM, 0.70),
    ),
    WAXY_EXOSKELETON(
        RequiresTag(EffectTag.MOTILE),
        MaintenanceCost(0.008),
        NumericMultiplier(NumericTarget.MIN_OPTIMUM_MOISTURE_MM, 0.35),
        NumericOffset(NumericTarget.MIN_OPTIMUM_TEMPERATURE_C, 2.0),
        NumericOffset(NumericTarget.MAX_OPTIMUM_TEMPERATURE_C, 5.0),
    ),
    ENLARGED_LUNGS(
        RequiresTag(EffectTag.MOTILE),
        MaintenanceCost(0.014),
        NumericMultiplier(NumericTarget.MAX_CONSUMPTION_RATE, 0.96),
        AltitudeToleranceEffect(maximumOptimalAltitudeMeters = 6_000.0, mortalityPerKilometerAbove = 0.22),
    ),
    HIGH_AFFINITY_HEMOGLOBIN(
        RequiresTag(EffectTag.MOTILE),
        MaintenanceCost(0.010),
        AltitudeToleranceEffect(maximumOptimalAltitudeMeters = 5_200.0, mortalityPerKilometerAbove = 0.25),
    ),
    ALPINE_CUSHION_GROWTH(
        RequiresTag(EffectTag.AUTOTROPHY),
        GrantsTag(EffectTag.WINTER_SURVIVAL),
        MaintenanceCost(0.008),
        NumericMultiplier(NumericTarget.AUTOTROPHY_GROWTH_RATE, 0.72),
        NumericMultiplier(NumericTarget.CARRYING_CAPACITY, 0.60),
        NumericOffset(NumericTarget.MIN_OPTIMUM_TEMPERATURE_C, -10.0),
        AltitudeToleranceEffect(maximumOptimalAltitudeMeters = 5_500.0, mortalityPerKilometerAbove = 0.20),
    ),
    OPEN_PLAINS_LOCOMOTION(
        RequiresTag(EffectTag.MOTILE),
        MaintenanceCost(0.006),
        NumericMultiplier(NumericTarget.MAX_CONSUMPTION_RATE, 1.05),
        NumericMultiplier(NumericTarget.INTERFERENCE, 0.75),
        HabitatPreferenceEffect(
            axis = HabitatAxis.CANOPY_CLOSURE,
            maximum = 0.25,
            mortalityAtExtreme = 1.10,
        ),
    ),
    WOODLAND_FORAGING(
        RequiresTag(EffectTag.FEEDING),
        MaintenanceCost(0.004),
        NumericMultiplier(NumericTarget.INTERFERENCE, 0.85),
        HabitatRefugeEffect(
            axis = HabitatAxis.CANOPY_CLOSURE,
            refugeStartsAt = 0.10,
            predatorPreferenceAtFullRefuge = 0.65,
        ),
        HabitatPreferenceEffect(
            axis = HabitatAxis.CANOPY_CLOSURE,
            maximum = 0.75,
            mortalityAtExtreme = 0.45,
        ),
    ),
    WETLAND_BREEDING(
        RequiresTag(EffectTag.MOTILE),
        MaintenanceCost(0.004),
        HabitatPreferenceEffect(
            axis = HabitatAxis.WETLAND_AVAILABILITY,
            minimum = 0.25,
            mortalityAtExtreme = 0.65,
        ),
        HabitatRefugeEffect(
            axis = HabitatAxis.WETLAND_AVAILABILITY,
            refugeStartsAt = 0.20,
            predatorPreferenceAtFullRefuge = 0.50,
        ),
    ),
    AQUATIC_LARVAE(
        RequiresTag(EffectTag.MOTILE),
        MaintenanceCost(0.006),
        NumericMultiplier(NumericTarget.MORTALITY_RATE, 0.85),
        HabitatRefugeEffect(
            axis = HabitatAxis.WETLAND_AVAILABILITY,
            refugeStartsAt = 0.25,
            predatorPreferenceAtFullRefuge = 0.30,
        ),
    ),
    PERMEABLE_SKIN(
        RequiresTag(EffectTag.MOTILE),
        NumericMultiplier(NumericTarget.MIN_OPTIMUM_MOISTURE_MM, 1.50),
    ),
    PROJECTILE_TONGUE(
        RequiresTag(EffectTag.FEEDING),
        GrantsTag(EffectTag.AERIAL_PREY_CAPTURE),
        MaintenanceCost(0.006),
        NumericMultiplier(NumericTarget.MAX_CONSUMPTION_RATE, 1.12),
    ),
    BOG_ROOTS(
        RequiresTag(EffectTag.SESSILE),
        NumericMultiplier(NumericTarget.MIN_OPTIMUM_MOISTURE_MM, 1.50),
    ),
    HUMP_FAT(
        RequiresTag(EffectTag.MOTILE),
        MaintenanceCost(0.018),
        NumericMultiplier(NumericTarget.STARVATION_MORTALITY_RATE, 0.55),
        NumericOffset(NumericTarget.MIN_OPTIMUM_TEMPERATURE_C, 2.0),
        NumericMultiplier(NumericTarget.MIN_OPTIMUM_MOISTURE_MM, 0.55),
        NumericOffset(NumericTarget.MAX_OPTIMUM_TEMPERATURE_C, 4.0),
    ),
    RUMINANT_STOMACH(
        RequiresTag(EffectTag.FEEDING),
        MaintenanceCost(0.005),
        FeedsOn(FoodResource.COARSE_GRASS, preference = 1.00),
        NumericMultiplier(NumericTarget.ASSIMILATION_EFFICIENCY, 1.15),
        NumericMultiplier(NumericTarget.MAX_CONSUMPTION_RATE, 0.90),
    ),
    HINDGUT_FERMENTATION(
        RequiresTag(EffectTag.FEEDING),
        MaintenanceCost(0.012),
        NumericMultiplier(NumericTarget.ASSIMILATION_EFFICIENCY, 1.10),
        NumericMultiplier(NumericTarget.MAX_CONSUMPTION_RATE, 1.05),
    ),
    LONG_NECK(
        RequiresTag(EffectTag.FEEDING),
        MaintenanceCost(0.018),
        FeedsOn(FoodResource.CANOPY_FOLIAGE, preference = 1.15),
    ),
    PREHENSILE_TRUNK(
        RequiresTag(EffectTag.FEEDING),
        MaintenanceCost(0.015),
        FeedsOn(FoodResource.LOW_FOLIAGE, preference = 1.05),
        FeedsOn(FoodResource.CANOPY_FOLIAGE, preference = 1.10),
        FeedsOn(FoodResource.FRUIT, preference = 1.10),
    ),
    TUSKS(
        MaintenanceCost(0.015),
        DefenseEffect(
            defaultPreferenceMultiplier = 0.55,
            ignoredWhenFeederSizeAdvantageAtLeast = 2,
        ),
    ),
    SNOW_CAMOUFLAGE(
        ExclusiveEffect(ExclusiveGroup.CAMOUFLAGE),
        MaintenanceCost(0.008),
        CamouflageEffect(CamouflageHabitat.SNOW),
    ),
    DESERT_CAMOUFLAGE(
        ExclusiveEffect(ExclusiveGroup.CAMOUFLAGE),
        MaintenanceCost(0.008),
        CamouflageEffect(CamouflageHabitat.DESERT),
    ),
    GRASSLAND_CAMOUFLAGE(
        ExclusiveEffect(ExclusiveGroup.CAMOUFLAGE),
        MaintenanceCost(0.008),
        CamouflageEffect(CamouflageHabitat.GRASSLAND),
    ),
    FOREST_CAMOUFLAGE(
        ExclusiveEffect(ExclusiveGroup.CAMOUFLAGE),
        MaintenanceCost(0.008),
        CamouflageEffect(CamouflageHabitat.FOREST),
    ),
    ADAPTIVE_CAMOUFLAGE(
        ExclusiveEffect(ExclusiveGroup.CAMOUFLAGE),
        MaintenanceCost(0.040),
        CamouflageEffect(
            habitat = CamouflageHabitat.ADAPTIVE,
            predatorMultiplierWhenMatched = 1.12,
            preyMultiplierWhenMatched = 0.65,
            predatorMultiplierWhenMismatched = 1.12,
            preyMultiplierWhenMismatched = 0.65,
        ),
    ),
    BRIGHT_MATING_PLUMAGE(
        MaintenanceCost(0.012),
        ConspicuousDisplayEffect(predatorHuntingMultiplier = 0.78, preyVulnerabilityMultiplier = 1.55),
    ),
    EXTRAVAGANT_DISPLAY_PLUMES(
        MaintenanceCost(0.025),
        ConspicuousDisplayEffect(predatorHuntingMultiplier = 0.65, preyVulnerabilityMultiplier = 1.85),
    ),
    SCALES(
        RequiresTag(EffectTag.MOTILE),
        MaintenanceCost(0.012),
        NumericOffset(NumericTarget.MIN_OPTIMUM_TEMPERATURE_C, 2.0),
        NumericOffset(NumericTarget.MAX_OPTIMUM_TEMPERATURE_C, 4.0),
        NumericMultiplier(NumericTarget.MIN_OPTIMUM_MOISTURE_MM, 0.80),
        DefenseEffect(defaultPreferenceMultiplier = 0.85),
    ),
    HIBERNATION(
        RequiresTag(EffectTag.MOTILE),
        MaintenanceCost(0.015),
        NumericMultiplier(NumericTarget.MAX_CONSUMPTION_RATE, 0.90),
        NumericMultiplier(NumericTarget.METABOLIC_DEMAND_RATE, 0.40),
        NumericMultiplier(NumericTarget.STARVATION_MORTALITY_RATE, 0.35),
        NumericOffset(NumericTarget.MIN_OPTIMUM_TEMPERATURE_C, -4.0),
        NumericMultiplier(NumericTarget.CLIMATE_STRESS_SENSITIVITY, 0.85),
    ),
    SEASONAL_MIGRATION(
        RequiresTag(EffectTag.MOTILE),
//        MaintenanceCost(0.035),
//        NumericMultiplier(NumericTarget.CLIMATE_STRESS_SENSITIVITY, 0.65),
    ),
    INSULATING_FEATHERS(
        RequiresTag(EffectTag.MOTILE),
        MaintenanceCost(0.008),
        NumericOffset(NumericTarget.MIN_OPTIMUM_TEMPERATURE_C, -6.0),
        NumericOffset(NumericTarget.MAX_OPTIMUM_TEMPERATURE_C, -2.0),
    ),
    STREAMLINED_BODY(
        RequiresTag(EffectTag.MOTILE),
        MaintenanceCost(0.012),
        NumericMultiplier(NumericTarget.MAX_CONSUMPTION_RATE, 1.10),
        NumericMultiplier(NumericTarget.INTERFERENCE, 0.85),
    ),
    ECHOLOCATION(
        RequiresTag(EffectTag.MOTILE),
        MaintenanceCost(0.025),
        NumericMultiplier(NumericTarget.MIN_OPTIMUM_INSOLATION_W_M2, 0.0),
    ),
    ELECTRORECEPTION(
        RequiresTag(EffectTag.MOTILE),
        MaintenanceCost(0.018),
        NumericMultiplier(NumericTarget.MIN_OPTIMUM_INSOLATION_W_M2, 0.0),
    ),
    BIOLUMINESCENT_LURE(
        RequiresTag(EffectTag.FEEDING),
        MaintenanceCost(0.025),
        NumericMultiplier(NumericTarget.MIN_OPTIMUM_INSOLATION_W_M2, 0.0),
        NumericMultiplier(NumericTarget.MAX_CONSUMPTION_RATE, 1.10),
    ),
    HEAT_SENSING_PITS(
        RequiresTag(EffectTag.MOTILE),
        MaintenanceCost(0.012),
        NumericMultiplier(NumericTarget.MIN_OPTIMUM_INSOLATION_W_M2, 0.30),
    ),
    LOW_BASAL_METABOLISM(
        RequiresTag(EffectTag.MOTILE),
        NumericMultiplier(NumericTarget.MAX_CONSUMPTION_RATE, 0.35),
        NumericMultiplier(NumericTarget.MORTALITY_RATE, 0.35),
        NumericMultiplier(NumericTarget.METABOLIC_DEMAND_RATE, 0.35),
        NumericMultiplier(NumericTarget.STARVATION_MORTALITY_RATE, 0.50),
    ),
    ANTIFREEZE_PROTEINS(
        RequiresTag(EffectTag.MOTILE),
        MaintenanceCost(0.015),
        NumericOffset(NumericTarget.MIN_OPTIMUM_TEMPERATURE_C, -6.0),
        NumericOffset(NumericTarget.MAX_OPTIMUM_TEMPERATURE_C, -1.0),
    ),
    WATERLOGGING_SENSITIVE_ROOTS(
        RequiresTag(EffectTag.SESSILE),
        NumericUpperBound(NumericTarget.MAX_OPTIMUM_MOISTURE_MM, 120.0),
    );

    /** Immutable effects declared by this named trait. */
    override val effects: List<TraitEffect> = effectList.toList()
}

/** Taxonomic order used as the production-level pool assigned to a BiotaDistribution. */
enum class TaxonomicOrder {
    POALES, FABALES, ASTERALES, ERICALES, ROSALES, SAPINDALES, ARECALES,
    FAGALES, MALVALES, PINALES, MALPIGHIALES, CARYOPHYLLALES,
    DINOPHYSIALES, SACOGLOSSA, EUPHAUSIACEA, LAGOMORPHA, RODENTIA,
    ORTHOPTERA, LEPIDOPTERA, COLEOPTERA, DIPTERA, ANURA, SQUAMATA,
    CARNIVORA, STRIGIFORMES, ACCIPITRIFORMES, PASSERIFORMES, COLUMBIFORMES,
    GALLIFORMES, PICIFORMES, ARTIODACTYLA, PRIMATES, PILOSA, PERISSODACTYLA,
    PROBOSCIDEA, CROCODILIA, SPHENISCIFORMES, SCORPIONES, DIPROTODONTIA,
    CETACEA, LAMNIFORMES, SCOMBRIFORMES, MYLIOBATIFORMES, LOPHIIFORMES,
    TESTUDINES, ORECTOLOBIFORMES, PROCELLARIIFORMES,
}

/** Stable catalog membership used to construct blueprints and distribution pools. */
val speciesIdsByTaxonomicOrder: Map<TaxonomicOrder, Set<String>> = mapOf(
    TaxonomicOrder.POALES to setOf("grass", "bamboo", "reeds"),
    TaxonomicOrder.FABALES to setOf("clover", "mesquite", "acacia"),
    TaxonomicOrder.ASTERALES to setOf("wildflowers", "edelweiss"),
    TaxonomicOrder.ERICALES to setOf("shrub", "berry_bush"),
    TaxonomicOrder.ROSALES to setOf("apple_tree", "fig_tree"),
    TaxonomicOrder.SAPINDALES to setOf("mango_tree"),
    TaxonomicOrder.ARECALES to setOf("date_palm"),
    TaxonomicOrder.FAGALES to setOf("oak"),
    TaxonomicOrder.MALVALES to setOf("kapok_tree"),
    TaxonomicOrder.PINALES to setOf("spruce"),
    TaxonomicOrder.MALPIGHIALES to setOf("dwarf_willow"),
    TaxonomicOrder.CARYOPHYLLALES to setOf("cactus", "sundew"),
    TaxonomicOrder.DINOPHYSIALES to setOf("phytoplankton"),
    TaxonomicOrder.SACOGLOSSA to setOf("solar_slug"),
    TaxonomicOrder.EUPHAUSIACEA to setOf("krill"),
    TaxonomicOrder.LAGOMORPHA to setOf("rabbit", "hare"),
    TaxonomicOrder.RODENTIA to setOf("mouse", "vole", "beaver", "capybara", "jerboa"),
    TaxonomicOrder.ORTHOPTERA to setOf("grasshopper", "cricket"),
    TaxonomicOrder.LEPIDOPTERA to setOf("caterpillar"),
    TaxonomicOrder.COLEOPTERA to setOf("beetle"),
    TaxonomicOrder.DIPTERA to setOf("fruit_fly"),
    TaxonomicOrder.ANURA to setOf("frog"),
    TaxonomicOrder.SQUAMATA to setOf("snake", "anaconda", "monitor_lizard"),
    TaxonomicOrder.CARNIVORA to setOf(
        "weasel", "fox", "coyote", "lynx", "wolf", "bear", "fennec_fox",
        "jaguar", "lion", "hyena", "tiger", "giant_panda", "wolverine",
        "arctic_fox", "polar_bear", "harbor_seal", "walrus",
    ),
    TaxonomicOrder.STRIGIFORMES to setOf("owl"),
    TaxonomicOrder.ACCIPITRIFORMES to setOf("hawk", "eagle"),
    TaxonomicOrder.PASSERIFORMES to setOf("sparrow"),
    TaxonomicOrder.COLUMBIFORMES to setOf("pigeon"),
    TaxonomicOrder.GALLIFORMES to setOf("quail"),
    TaxonomicOrder.PICIFORMES to setOf("toucan"),
    TaxonomicOrder.ARTIODACTYLA to setOf(
        "deer", "boar", "dromedary", "bactrian_camel", "oryx", "giraffe",
        "wildebeest", "moose", "caribou", "ibex", "bison", "saiga",
        "musk_ox", "yak",
    ),
    TaxonomicOrder.PRIMATES to setOf("gorilla"),
    TaxonomicOrder.PILOSA to setOf("sloth"),
    TaxonomicOrder.PERISSODACTYLA to setOf("tapir", "zebra"),
    TaxonomicOrder.PROBOSCIDEA to setOf("elephant", "mammoth"),
    TaxonomicOrder.CROCODILIA to setOf("crocodile"),
    TaxonomicOrder.SPHENISCIFORMES to setOf("emperor_penguin"),
    TaxonomicOrder.SCORPIONES to setOf("scorpion"),
    TaxonomicOrder.DIPROTODONTIA to setOf("kangaroo"),
    TaxonomicOrder.CETACEA to setOf("orca", "blue_whale", "dolphin"),
    TaxonomicOrder.LAMNIFORMES to setOf("great_white_shark"),
    TaxonomicOrder.SCOMBRIFORMES to setOf("tuna"),
    TaxonomicOrder.MYLIOBATIFORMES to setOf("manta_ray"),
    TaxonomicOrder.LOPHIIFORMES to setOf("anglerfish"),
    TaxonomicOrder.TESTUDINES to setOf("sea_turtle"),
    TaxonomicOrder.ORECTOLOBIFORMES to setOf("whale_shark"),
    TaxonomicOrder.PROCELLARIIFORMES to setOf("albatross"),
)

/** Reverse lookup required while constructing a blueprint from its stable catalog id. */
val taxonomicOrderBySpeciesId: Map<String, TaxonomicOrder> =
    speciesIdsByTaxonomicOrder.entries.flatMap { (order, ids) -> ids.map { it to order } }.toMap()

/** Trait-only, production-facing description of a procedurally generated species. */
data class SpeciesBlueprint(
    /** Stable identifier used in biomass maps and food-web references. */
    val id: String,
    /** Taxonomic order controlling which BiotaDistribution pools may generate this species. */
    val taxonomicOrder: TaxonomicOrder,
    /** Complete set of size, niche, body, behavior, and defense traits. */
    val traits: Set<SpeciesTrait>,
) {
    init {
        require(id.isNotBlank()) { "Species id must not be blank" }
        require(traits.filterIsInstance<SizeClass>().size == 1) {
            id + " must have exactly one size-class trait"
        }
    }

    /** The blueprint's required and unique size-class trait. */
    val sizeClass: SizeClass
        get() = traits.filterIsInstance<SizeClass>().single()

    /** Flattened effect descriptors contributed by all traits. */
    val effects: List<TraitEffect>
        get() = traits.flatMap { it.effects }

    /** Capability and countermeasure tags granted by the flattened effects. */
    val effectTags: Set<EffectTag>
        get() = effects.filterIsInstance<GrantsTag>().map { it.tag }.toSet()

    /** Returns whether this blueprint explicitly contains a named trait. */
    fun has(trait: Trait): Boolean = trait in traits

    /** Returns whether its effects grant a capability or countermeasure tag. */
    fun hasTag(tag: EffectTag): Boolean = tag in effectTags
}

/** Convenience constructor that deduplicates a vararg collection of traits. */
fun speciesBlueprint(id: String, vararg traits: SpeciesTrait): SpeciesBlueprint =
    SpeciesBlueprint(id, taxonomicOrderBySpeciesId.getValue(id), traits.toSet())

/** Constructor for procedurally generated species whose id is not in the fixed catalog mapping. */
fun speciesBlueprint(
    id: String,
    taxonomicOrder: TaxonomicOrder,
    vararg traits: SpeciesTrait,
): SpeciesBlueprint = SpeciesBlueprint(id, taxonomicOrder, traits.toSet())

/** One independently contested edible resource exposed by a species. */
data class FoodSource(
    /** Species producing or embodying the food. */
    val preyId: String,
    /** Exact tissue, product, or prey-body resource being consumed. */
    val resource: FoodResource,
)

/** Compiled feeding relationship between one feeder and one exact food source. */
data class DietEntry(
    /** Relative targeting weight before prey abundance is considered. */
    val preference: Double,
    /** Prey density at which this food becomes readily exploitable. */
    val halfSaturation: Double,
    /** Fraction of ingestion removed from the food species' standing biomass. */
    val preyLossFraction: Double = 1.0,
    /** Renewable production cap per producer biomass per year, or null for standing biomass. */
    val renewableProductionRate: Double? = null,
) {
    init {
        require(preference > 0.0)
        require(halfSaturation > 0.0)
        require(preyLossFraction in 0.0..1.0)
        require(renewableProductionRate == null || renewableProductionRate > 0.0)
    }
}

/** Numerical parameters for biomass gained through autotrophy. */
data class AutotrophyParameters(
    /** Intrinsic logistic growth rate per year. */
    val growthRate: Double,
    /** Default carrying-capacity density before simulation overrides and seasons. */
    val baseCarryingCapacity: Double,
    /** Fractional amplitude of seasonal carrying-capacity change. */
    val seasonalAmplitude: Double,
    /** Position of the seasonal peak within a year. */
    val seasonalPhase: Double,
    /** Insolation in W/m² at which photosynthesis reaches half its light-limited maximum. */
    val photosynthesisHalfSaturationWm2: Double,
)

/** Numerical parameters for feeding, metabolism, mortality, and crowding. */
data class FeedingParameters(
    /** Maximum food biomass ingested per feeder biomass per year. */
    val maxConsumptionRate: Double,
    /** Fraction of ingested biomass converted into feeder biomass. */
    val assimilationEfficiency: Double,
    /** Baseline fractional biomass mortality per year. */
    val mortalityRate: Double,
    /** Assimilated food biomass required per feeder biomass per year. */
    val metabolicDemandRate: Double,
    /** Additional fractional mortality per year when metabolic demand is completely unmet. */
    val starvationMortalityRate: Double,
    /** Strength of quadratic self-limitation per unit land area. */
    val densityDependence: Double,
    /** Strength of feeder interference in the functional response. */
    val interference: Double,
    /** Shape exponent of the low-prey feeding response. */
    val responseExponent: Double,
)

/** Trait-derived climate interval in which a species has no climate mortality. */
data class ClimateNiche(
    /** Lowest temperature with no climate stress, in degrees Celsius. */
    val minOptimalTemperatureC: Double,
    /** Highest temperature with no climate stress, in degrees Celsius. */
    val maxOptimalTemperatureC: Double,
    /** Lowest monthly precipitation with no moisture stress, in millimetres. */
    val minOptimalMoistureMm: Double,
    /** Optional upper no-stress precipitation bound; null means no upper bound. */
    val maxOptimalMoistureMm: Double?,
    /** Lowest no-stress mean insolation in W/m². */
    val minOptimalInsolationWm2: Double,
    /** Highest no-stress mean insolation in W/m². */
    val maxOptimalInsolationWm2: Double,
    /** Multiplier controlling additional mortality outside the optimum interval. */
    val stressSensitivity: Double,
) {
    init {
        require(minOptimalTemperatureC < maxOptimalTemperatureC)
        require(minOptimalMoistureMm >= 0.0)
        require(maxOptimalMoistureMm == null || maxOptimalMoistureMm > minOptimalMoistureMm)
        require(minOptimalInsolationWm2 >= 0.0)
        require(maxOptimalInsolationWm2 > minOptimalInsolationWm2)
        require(stressSensitivity >= 0.0)
    }
}

/** Default climate used by the notebook examples: mild, wet oceanic temperate seasons. */
val oceanicTemperateClimate = ClimateDatum(
    tileId = -1,
    months = listOf(
        ClimateDatumSample(6.0, 60.0, 115.0),
        ClimateDatumSample(6.5, 90.0, 100.0),
        ClimateDatumSample(8.0, 130.0, 90.0),
        ClimateDatumSample(10.0, 180.0, 80.0),
        ClimateDatumSample(13.0, 220.0, 75.0),
        ClimateDatumSample(16.0, 250.0, 70.0),
        ClimateDatumSample(18.0, 260.0, 70.0),
        ClimateDatumSample(18.0, 220.0, 75.0),
        ClimateDatumSample(15.5, 160.0, 85.0),
        ClimateDatumSample(12.0, 110.0, 100.0),
        ClimateDatumSample(9.0, 70.0, 115.0),
        ClimateDatumSample(7.0, 50.0, 120.0),
    ),
)

/** Builds an illustrative January-to-December climate in native ecology units. */
fun sampleClimate(
    tileId: Int,
    temperaturesC: List<Double>,
    insolationsWm2: List<Double>,
    precipitationMm: List<Double>,
): ClimateDatum {
    require(temperaturesC.size == 12)
    require(insolationsWm2.size == 12)
    require(precipitationMm.size == 12)
    return ClimateDatum(
        tileId = tileId,
        months = temperaturesC.indices.map { month ->
            ClimateDatumSample(
                temperaturesC[month],
                insolationsWm2[month],
                precipitationMm[month],
            )
        },
    )
}

/** Hot subtropical desert with intense light and very sparse rainfall. */
val desertClimate = sampleClimate(
    tileId = -2,
    temperaturesC = listOf(18.0, 20.0, 24.0, 29.0, 33.0, 36.0, 38.0, 37.0, 34.0, 29.0, 23.0, 19.0),
    insolationsWm2 = listOf(220.0, 245.0, 280.0, 310.0, 330.0, 345.0, 340.0, 325.0, 300.0, 270.0, 235.0, 215.0),
    precipitationMm = listOf(5.0, 4.0, 4.0, 2.0, 1.0, 0.5, 1.0, 2.0, 3.0, 5.0, 6.0, 6.0),
)

/** Tropical savanna with a pronounced summer wet season and winter drought. */
val savannaClimate = sampleClimate(
    tileId = -3,
    temperaturesC = listOf(24.0, 25.0, 27.0, 29.0, 29.0, 28.0, 27.0, 27.0, 28.0, 28.0, 26.0, 24.0),
    insolationsWm2 = listOf(235.0, 250.0, 270.0, 280.0, 265.0, 235.0, 220.0, 225.0, 245.0, 265.0, 255.0, 235.0),
    precipitationMm = listOf(8.0, 12.0, 25.0, 65.0, 130.0, 190.0, 220.0, 185.0, 110.0, 45.0, 15.0, 8.0),
)

/** Equatorial rainforest with warm temperatures and abundant rain year-round. */
val jungleClimate = sampleClimate(
    tileId = -4,
    temperaturesC = listOf(26.0, 26.0, 26.5, 26.5, 26.5, 26.0, 25.5, 26.0, 26.5, 26.5, 26.0, 26.0),
    insolationsWm2 = listOf(205.0, 215.0, 220.0, 215.0, 200.0, 185.0, 190.0, 205.0, 220.0, 225.0, 215.0, 205.0),
    precipitationMm = listOf(240.0, 220.0, 260.0, 285.0, 300.0, 245.0, 210.0, 205.0, 225.0, 275.0, 290.0, 260.0),
)

/** Continental boreal forest with a long cold winter and short mild summer. */
val borealClimate = sampleClimate(
    tileId = -5,
    temperaturesC = listOf(-18.0, -15.0, -8.0, 1.0, 9.0, 15.0, 18.0, 15.0, 8.0, 0.0, -9.0, -16.0),
    insolationsWm2 = listOf(25.0, 55.0, 110.0, 175.0, 225.0, 255.0, 245.0, 190.0, 125.0, 65.0, 30.0, 15.0),
    precipitationMm = listOf(25.0, 20.0, 22.0, 28.0, 40.0, 60.0, 75.0, 65.0, 50.0, 38.0, 30.0, 25.0),
)

/** Arctic tundra with a brief cool growing season and low precipitation. */
val tundraClimate = sampleClimate(
    tileId = -6,
    temperaturesC = listOf(-26.0, -25.0, -20.0, -11.0, -3.0, 4.0, 8.0, 6.0, 1.0, -8.0, -18.0, -24.0),
    insolationsWm2 = listOf(2.0, 18.0, 75.0, 145.0, 205.0, 235.0, 215.0, 160.0, 90.0, 32.0, 5.0, 0.5),
    precipitationMm = listOf(10.0, 8.0, 8.0, 9.0, 12.0, 18.0, 25.0, 24.0, 18.0, 14.0, 11.0, 10.0),
)

/** Permanent ice cap with no month above freezing and polar-light seasonality. */
val iceCapClimate = sampleClimate(
    tileId = -7,
    temperaturesC = listOf(-38.0, -40.0, -38.0, -31.0, -23.0, -17.0, -14.0, -17.0, -24.0, -31.0, -35.0, -37.0),
    insolationsWm2 = listOf(0.0, 2.0, 35.0, 100.0, 165.0, 205.0, 190.0, 125.0, 55.0, 10.0, 0.0, 0.0),
    precipitationMm = listOf(5.0, 4.0, 4.0, 4.0, 5.0, 7.0, 9.0, 8.0, 7.0, 6.0, 5.0, 5.0),
)

/** Drop-in named climate presets for notebook ecosystem experiments. */
val sampleClimates: Map<String, ClimateDatum> = mapOf(
    "desert" to desertClimate,
    "savanna" to savannaClimate,
    "jungle" to jungleClimate,
    "boreal" to borealClimate,
    "tundra" to tundraClimate,
    "ice_cap" to iceCapClimate,
)

/** Fully compiled numerical species consumed by the simulation solver. */
data class SpeciesDefinition(
    /** Trait-only source from which every other field was derived. */
    val blueprint: SpeciesBlueprint,
    /** Derived autotrophy capability, or null when the species lacks it. */
    val autotrophy: AutotrophyParameters?,
    /** Derived feeding capability, or null when the species lacks it. */
    val feeding: FeedingParameters?,
    /** Trait- and size-derived temperature and moisture tolerances. */
    val climateNiche: ClimateNiche,
    /** Baseline plus additive trait upkeep paid as fractional biomass loss per year. */
    val maintenanceRate: Double,
    /** Derived food-web entries keyed by prey species and exact resource. */
    val diet: Map<FoodSource, DietEntry>,
) {
    /** Stable species id delegated from the blueprint. */
    val id: String get() = blueprint.id

    /** Taxonomic order delegated from the blueprint for distribution-level filtering. */
    val taxonomicOrder: TaxonomicOrder get() = blueprint.taxonomicOrder

    /** Original immutable trait set delegated from the blueprint. */
    val traits: Set<SpeciesTrait> get() = blueprint.traits

    /** Flattened declarative effects delegated from the blueprint. */
    val effects: List<TraitEffect> get() = blueprint.effects

    /** Unique body-size class delegated from the blueprint. */
    val sizeClass: SizeClass get() = blueprint.sizeClass

    /** Biomass represented by one individual of this species. */
    val individualBiomass: Double get() = sizeClass.individualBiomass

    init {
        require(autotrophy != null || feeding != null) {
            id + " needs at least one biomass-acquisition capability"
        }
        require(feeding == null || diet.isNotEmpty()) {
            "Feeding species " + id + " needs at least one derived food"
        }
        require(feeding != null || diet.isEmpty()) {
            "Non-feeding species " + id + " should not have a diet"
        }
    }
}

/** Returns one resource-specific diet entry, if this species can consume it. */
fun SpeciesDefinition.dietEntry(
    preyId: String,
    resource: FoodResource,
): DietEntry? = diet[FoodSource(preyId, resource)]

/** Strongest preference across all resources supplied by one prey species. */
fun SpeciesDefinition.maximumPreferenceFor(preyId: String): Double = diet
    .filterKeys { it.preyId == preyId }
    .values
    .maxOfOrNull { it.preference } ?: 0.0

/** One possible feeder-resource pathway before constrained allocation. */
data class FeedingOpportunity(
    /** Species attempting to consume this food source. */
    val feeder: String,
    /** Exact resource pathway being considered. */
    val source: FoodSource,
    /** Relative abundance-and-preference weight used during reallocation. */
    val signal: Double,
    /** Maximum ingestion rate allowed by this pathway's own Type III response. */
    val pathwayCapacity: Double,
    /** Fraction of accepted ingestion removed from standing prey biomass. */
    val preyLossFraction: Double,
    /** Fraction of accepted ingestion assimilated by the feeder. */
    val assimilationEfficiency: Double,
    /** Shared renewable production rate, or null for standing-biomass foods. */
    val renewableProductionRate: Double?,
)

/** One instantaneous transfer of biomass caused by feeding. */
data class FeedingFlux(
    /** Species receiving assimilated biomass. */
    val feeder: String,
    /** Exact species resource supplying the ingestion. */
    val source: FoodSource,
    /** Food biomass processed by the feeder. */
    val ingested: Double,
    /** Standing biomass actually removed from the prey species. */
    val preyLoss: Double,
    /** Ingested biomass converted into new feeder biomass. */
    val assimilated: Double,
    /** Renewable production limit used to cap all consumers sharing this source. */
    val renewableProductionRate: Double? = null,
) {
    /** Species whose standing biomass pays any prey loss. */
    val prey: String get() = source.preyId
}

/** Returns whether this compiled species participates in a sessile producer size guild. */
fun SpeciesDefinition.isSessileProducer(): Boolean =
    autotrophy != null && blueprint.hasTag(EffectTag.SESSILE)

/** Shared seasonal-capacity pattern compiled for one sessile size guild. */
data class SeasonalCapacityPattern(
    /** Fractional seasonal oscillation of the shared capacity. */
    val amplitude: Double,
    /** Average position of the guild's seasonal peak within a year. */
    val phase: Double,
)

/** Cosine similarity of two consumers' available diet-preference vectors. */
fun consumerNicheOverlap(
    first: SpeciesDefinition,
    second: SpeciesDefinition,
    availableFoodSources: Set<FoodSource>,
): Double {
    if (first.id == second.id) return 1.0
    var dotProduct = 0.0
    var firstSquared = 0.0
    var secondSquared = 0.0
    for (source in availableFoodSources) {
        val firstWeight = first.diet[source]?.preference ?: 0.0
        val secondWeight = second.diet[source]?.preference ?: 0.0
        dotProduct += firstWeight * secondWeight
        firstSquared += firstWeight * firstWeight
        secondSquared += secondWeight * secondWeight
    }
    val denominator = sqrt(firstSquared * secondSquared)
    return if (denominator > 0.0) {
        (dotProduct / denominator).coerceIn(0.0, 1.0)
    } else 0.0
}

/** Precomputes static diet overlap used to couple consumers' feeding interference. */
fun deriveConsumerNicheOverlaps(
    species: List<SpeciesDefinition>,
): Map<String, Map<String, Double>> {
    val availableSpeciesIds = species.map { it.id }.toSet()
    val consumers = species.filter { it.feeding != null }
    val availableFoodSources = consumers
        .flatMap { it.diet.keys }
        .filter { it.preyId in availableSpeciesIds }
        .toSet()
    return consumers.associate { first ->
        first.id to consumers.associate { second ->
            second.id to consumerNicheOverlap(first, second, availableFoodSources)
        }
    }
}

/** Immutable configuration and compiled species for one isolated tile simulation. */
data class EcosystemModel(
    /** Species participating in this tile's ecosystem. */
    val species: List<SpeciesDefinition>,
    /** Whether carrying capacities vary seasonally. */
    val seasonsEnabled: Boolean = true,
    /** Tile land area; biomass totals scale with this value. */
    val landArea: Double = 1.0,
    /** Repeating monthly climate forcing; defaults to the notebook's oceanic biome. */
    val climate: ClimateDatum = oceanicTemperateClimate,
    /** Tile elevation above sea level; local temperature uses a 6.5 C/km lapse rate. */
    val altitudeMeters: Double = 500.0,
    /** Optional shared carrying-capacity density overrides keyed by sessile size class. */
    val sessileCarryingCapacityDensities: Map<SizeClass, Double> = emptyMap(),
) {
    /** Sessile autotrophs grouped into the size guilds that share capacity. */
    val sessileProducerGroups: Map<SizeClass, List<SpeciesDefinition>> =
        species.filter { it.isSessileProducer() }.groupBy { it.sizeClass }

    /** Fast lookup used by dynamic predation and habitat effects. */
    val speciesById: Map<String, SpeciesDefinition> = species.associateBy { it.id }

    /** Shared seasonal pattern for each present sessile size guild. */
    val sessileSeasonality: Map<SizeClass, SeasonalCapacityPattern> =
        sessileProducerGroups.mapValues { (_, definitions) ->
            SeasonalCapacityPattern(
                amplitude = definitions.map { it.autotrophy!!.seasonalAmplitude }.average(),
                phase = definitions.map { it.autotrophy!!.seasonalPhase }.average(),
            )
        }

    /** Static pairwise diet overlap coefficients used only in feeding interference. */
    val consumerNicheOverlaps: Map<String, Map<String, Double>> =
        deriveConsumerNicheOverlaps(species)

    init {
        require(landArea > 0.0) { "Land area must be positive" }
        require(altitudeMeters in -500.0..9_000.0) { "Altitude must be between -500 m and 9000 m" }
        require(sessileCarryingCapacityDensities.values.all { it > 0.0 }) {
            "Sessile carrying-capacity densities must be positive"
        }
    }

    /** Returns the simulation override for a size guild, or its size-derived default. */
    fun baseSessileCarryingCapacityDensity(sizeClass: SizeClass): Double =
        sessileCarryingCapacityDensities[sizeClass]
            ?: sizeClass.defaultSessileCarryingCapacityDensity

    /** Returns the shared, seasonally adjusted capacity density of a sessile size guild. */
    fun sessileCarryingCapacityDensity(sizeClass: SizeClass, year: Double): Double {
        val pattern = sessileSeasonality[sizeClass] ?: SeasonalCapacityPattern(
            amplitude = 0.18,
            phase = 0.14 + 0.025 * sizeClass.rank,
        )
        val seasonalOffset = if (seasonsEnabled) {
            pattern.amplitude * sin(2.0 * PI * (year - pattern.phase))
        } else 0.0
        return baseSessileCarryingCapacityDensity(sizeClass) * (1.0 + seasonalOffset)
    }
}

/** Samples climate at tile elevation, treating ClimateDatum temperature as sea-level forcing. */
fun EcosystemModel.localClimateAt(year: Double): ClimateDatumSample {
    val sample = climate.sampleAt(year)
    return sample.copy(averageTemperature = sample.averageTemperature - 0.0065 * altitudeMeters)
}

/** Converts total biomass into an approximate (possibly fractional) individual count. */
fun approximateIndividuals(definition: SpeciesDefinition, biomass: Double): Double =
    biomass / definition.individualBiomass

/** Fraction of overhead light intercepted by canopies taller than an optional target rank. */
fun canopyClosure(
    year: Double,
    state: Biomass,
    model: EcosystemModel,
    tallerThanSizeRank: Int? = null,
): Double = model.species.sumOf { definition ->
    if (tallerThanSizeRank != null && definition.sizeClass.rank <= tallerThanSizeRank) {
        return@sumOf 0.0
    }
    val interception = definition.effects.filterIsInstance<CanopyEffect>()
        .sumOf { it.maximumLightInterception }
        .coerceAtMost(1.0)
    if (interception <= 0.0) return@sumOf 0.0
    val guildCapacity = model.sessileCarryingCapacityDensity(
        definition.sizeClass,
        year,
    ) * model.landArea
    interception * state.getValue(definition.id).coerceAtLeast(0.0) / guildCapacity
}.coerceIn(0.0, 0.98)

/** Remaining incident light at the top of a producer's own height layer. */
fun availableLightFraction(
    definition: SpeciesDefinition,
    year: Double,
    state: Biomass,
    model: EcosystemModel,
): Double = 1.0 - canopyClosure(year, state, model, definition.sizeClass.rank)

/** Current normalized value of one habitat dimension. */
fun habitatAxisValue(
    axis: HabitatAxis,
    year: Double,
    state: Biomass,
    model: EcosystemModel,
): Double = when (axis) {
    HabitatAxis.CANOPY_CLOSURE -> canopyClosure(year, state, model)
    HabitatAxis.WETLAND_AVAILABILITY ->
        (model.localClimateAt(year).precipitation / 100.0).coerceIn(0.0, 1.0)
}

/** Computes each habitat dimension once for reuse throughout one derivative evaluation. */
fun habitatAxisValues(
    year: Double,
    state: Biomass,
    model: EcosystemModel,
): Map<HabitatAxis, Double> = HabitatAxis.entries.associateWith { axis ->
    habitatAxisValue(axis, year, state, model)
}

/** Additional mortality from trait-declared habitat mismatch. */
fun habitatStressMortalityRate(
    definition: SpeciesDefinition,
    habitatValues: Map<HabitatAxis, Double>,
): Double = definition.effects.filterIsInstance<HabitatPreferenceEffect>().sumOf { effect ->
    val value = habitatValues.getValue(effect.axis)
    val mismatch = when {
        value < effect.minimum && effect.minimum > 0.0 ->
            (effect.minimum - value) / effect.minimum
        value > effect.maximum && effect.maximum < 1.0 ->
            (value - effect.maximum) / (1.0 - effect.maximum)
        else -> 0.0
    }
    effect.mortalityAtExtreme * mismatch.coerceIn(0.0, 1.0)
}

/** Dynamic multiplier on predation when prey can occupy a trait-declared refuge. */
fun habitatRefugePreferenceMultiplier(
    prey: SpeciesDefinition,
    habitatValues: Map<HabitatAxis, Double>,
): Double = prey.effects.filterIsInstance<HabitatRefugeEffect>().fold(1.0) { product, effect ->
    val availability = habitatValues.getValue(effect.axis)
    val refugeStrength = ((availability - effect.refugeStartsAt) /
        (1.0 - effect.refugeStartsAt)).coerceIn(0.0, 1.0)
    val multiplier = 1.0 - refugeStrength *
        (1.0 - effect.predatorPreferenceAtFullRefuge)
    product * multiplier
}

/** Fractional match between a camouflage pattern and this tile's current background. */
fun camouflageMatch(
    habitat: CamouflageHabitat,
    year: Double,
    state: Biomass,
    model: EcosystemModel,
    habitatValues: Map<HabitatAxis, Double>,
): Double {
    if (habitat == CamouflageHabitat.ADAPTIVE) return 1.0
    val climate = model.localClimateAt(year)
    val canopy = habitatValues.getValue(HabitatAxis.CANOPY_CLOSURE)
    val lowPlantCapacity = model.sessileProducerGroups[SizeClass.MINUSCULE]
        ?.sumOf { state.getValue(it.id).coerceAtLeast(0.0) }
        ?.div(model.sessileCarryingCapacityDensity(SizeClass.MINUSCULE, year) * model.landArea)
        ?.coerceIn(0.0, 1.0) ?: 0.0
    return when (habitat) {
        CamouflageHabitat.SNOW -> if (climate.averageTemperature < 0.0) {
            ((-climate.averageTemperature / 10.0).coerceIn(0.25, 1.0) *
                (climate.precipitation / 20.0).coerceIn(0.0, 1.0))
        } else 0.0
        CamouflageHabitat.DESERT ->
            (1.0 - climate.precipitation / 45.0).coerceIn(0.0, 1.0) * (1.0 - canopy)
        CamouflageHabitat.GRASSLAND -> lowPlantCapacity * (1.0 - canopy)
        CamouflageHabitat.FOREST -> canopy
        CamouflageHabitat.ADAPTIVE -> 1.0
    }
}

/** Hunting or vulnerability multiplier from camouflage and conspicuous displays. */
fun visualInteractionMultiplier(
    definition: SpeciesDefinition,
    asPredator: Boolean,
    year: Double,
    state: Biomass,
    model: EcosystemModel,
    habitatValues: Map<HabitatAxis, Double>,
): Double {
    val camouflage = definition.effects.filterIsInstance<CamouflageEffect>().fold(1.0) { product, effect ->
        val match = camouflageMatch(effect.habitat, year, state, model, habitatValues)
        val matched = if (asPredator) effect.predatorMultiplierWhenMatched else effect.preyMultiplierWhenMatched
        val mismatched = if (asPredator) effect.predatorMultiplierWhenMismatched else effect.preyMultiplierWhenMismatched
        product * (mismatched + match * (matched - mismatched))
    }
    return definition.effects.filterIsInstance<ConspicuousDisplayEffect>().fold(camouflage) { product, effect ->
        product * if (asPredator) effect.predatorHuntingMultiplier else effect.preyVulnerabilityMultiplier
    }
}


## Feeding and population change

The trait compiler creates each feeding species' resource-specific diet by comparing its niche, hunting, body, and defense traits against every possible prey blueprint. Dietary niche traits such as carnivore, frugivore, insectivore, and omnivore define access and preference only; physiological traits such as ruminant stomach and hindgut fermentation modify species-wide assimilation. Dedicated resources can represent unusually narrow relationships: bamboo culms expose a bamboo resource, and bamboo specialists consume that resource with high throughput but poor assimilation rather than inheriting access to every leafy plant. Fruiting independently exposes renewable fruit from wildflowers, shrubs, and apple, mango, fig, or date trees, so frugivores automatically gain those pathways whenever the corresponding producer is present. The numerical solver then uses a density-based Type III response for each exact species-resource pair.

For food that represents destructive herbivory or predation, ingestion removes prey biomass. Fruit and seeds instead define annual production caps per unit producer biomass; all consumers share those caps, and excess requests are proportionally scaled. A constrained iterative allocator redistributes unmet intake across every pathway with remaining capacity, including other renewable foods, without exceeding that pathway's own Type III ceiling. This allows omnivores to switch diets without converting unavailable fruit directly into unlimited predation. Small renewable-resource `preyLossFraction` values charge limited replacement costs to producer biomass. Production is available continuously through the year rather than being stored in a carry-over pool.

Every species also receives a trait-derived climate niche in the native ClimateDatumSample units: °C, W/m², and mm/month. ClimateDatum.sampleAt() interpolates the production climate directly. EcosystemModel defaults to 500 m altitude and applies a 6.5 °C/km lapse rate; altitude-tolerance traits separately reduce high-elevation hypoxia. Conditions outside a species' temperature, moisture, or insolation range add mortality, while autotrophic growth is powered jointly by interpolated insolation and moisture availability. The examples use a repeating oceanic temperate ClimateDatum by default.

Large-canopy traits intercept light in proportion to their current biomass and size-guild capacity, reducing photosynthesis only for shorter sessile producers. Animal habitat traits use the resulting canopy closure and a precipitation-derived seasonal wetland fraction. Habitat mismatch adds mortality, while woodland and aquatic-larval refuges dynamically reduce predator preference when the required cover is actually present. Fixed snow, desert, grassland, and forest camouflage improves hunting and concealment only where its background matches and becomes a liability elsewhere; adaptive camouflage always works but carries a much larger maintenance cost. Conspicuous mating colors make both hunting and avoiding predators harder.

Freezing temperatures now have an explicit seasonal-survival gate for sessile producers: dormant buds, underground meristems, needle leaves, or alpine cushion growth protect perennial tissue. Without one of those mechanisms, below-freezing months cause severe direct mortality even if another drought or heat adaptation shifted the broad temperature niche. Sparse coats and heat-radiating ears likewise impose direct frost exposure on warm-climate animals.

Burrowing reduces predation and buffers temperature, but now caps the no-stress rainfall range because saturated ground floods tunnels. Wetland-breeding amphibians gain a partial wetland refuge, aquatic larvae strengthen it, and projectile tongues specifically counter much of the evasion benefit of flying insect prey. Heavy insulation adds direct heat-retention mortality above its trait-specific threshold, so thick fur, blubber, and subcutaneous fat remain useful cold adaptations rather than universal bonuses. Sparse coats instead improve heat tolerance while creating severe direct frost exposure.

Prostrate woody growth traps enough boundary-layer heat to become directly harmful in hot climates, keeping tundra shrubs specialized even when their broad climate-stress multiplier is low.

Consumer background mortality is deliberately modest. Starvation is separate: current assimilated feeding is compared with the species' metabolic demand, and a nonlinear mortality penalty is added for any shortfall. At zero food the full starvation rate applies; warm-blooded species demand more food and starve faster, while fat reserves, low basal metabolism, and hibernation slow the decline.

In [ ]:
fun feedingFluxes(
    state: Biomass,
    model: EcosystemModel,
    year: Double = 0.0,
    habitatValues: Map<HabitatAxis, Double> = habitatAxisValues(year, state, model),
): List<FeedingFlux> {
    val opportunities = mutableListOf<FeedingOpportunity>()
    val targetIngestionByFeeder = mutableMapOf<String, Double>()
    val totalSignalByFeeder = mutableMapOf<String, Double>()

    for (definition in model.species) {
        val feeding = definition.feeding ?: continue
        val feederBiomass = state.getValue(definition.id)
        if (feederBiomass <= 0.0) continue
        val predatorVisualMultiplier = visualInteractionMultiplier(
            definition, true, year, state, model, habitatValues,
        )

        val foodSignals = definition.diet.mapValues { (source, diet) ->
            val preyBiomass = state[source.preyId]?.coerceAtLeast(0.0) ?: 0.0
            val preyDensity = preyBiomass / model.landArea
            val availableFoodDensity = preyDensity *
                (diet.renewableProductionRate ?: 1.0)
            val refugeMultiplier = if (source.resource == FoodResource.MOTILE_PREY) {
                model.speciesById[source.preyId]?.let { preyDefinition ->
                    habitatRefugePreferenceMultiplier(
                        preyDefinition,
                        habitatValues,
                    )
                } ?: 1.0
            } else 1.0
            val visualMultiplier = if (source.resource == FoodResource.MOTILE_PREY) {
                val preyDefinition = model.speciesById[source.preyId]
                predatorVisualMultiplier * if (preyDefinition != null) {
                    visualInteractionMultiplier(
                        preyDefinition, false, year, state, model, habitatValues,
                    )
                } else 1.0
            } else 1.0
            diet.preference * refugeMultiplier * visualMultiplier *
                (availableFoodDensity / diet.halfSaturation)
                    .pow(feeding.responseExponent)
        }.filterValues { it > 0.0 }
        val totalFoodSignal = foodSignals.values.sum()
        if (totalFoodSignal <= 0.0) continue

        val overlappingConsumerDensity = model.consumerNicheOverlaps
            .getValue(definition.id)
            .entries
            .sumOf { (competitorId, overlap) ->
                overlap * state.getValue(competitorId).coerceAtLeast(0.0)
            } / model.landArea
        val interferenceLoad = feeding.interference * overlappingConsumerDensity
        val maximumIngestion = feeding.maxConsumptionRate * feederBiomass
        val targetIngestion = maximumIngestion * totalFoodSignal /
            (1.0 + totalFoodSignal + interferenceLoad)
        targetIngestionByFeeder[definition.id] = targetIngestion
        totalSignalByFeeder[definition.id] = totalFoodSignal

        for ((source, signal) in foodSignals) {
            val diet = definition.diet.getValue(source)
            opportunities += FeedingOpportunity(
                feeder = definition.id,
                source = source,
                signal = signal,
                pathwayCapacity = maximumIngestion * signal /
                    (1.0 + signal + interferenceLoad),
                preyLossFraction = diet.preyLossFraction,
                assimilationEfficiency = feeding.assimilationEfficiency,
                renewableProductionRate = diet.renewableProductionRate,
            )
        }
    }

    val allocated = opportunities.associateWith { opportunity ->
        (targetIngestionByFeeder.getValue(opportunity.feeder) *
            opportunity.signal / totalSignalByFeeder.getValue(opportunity.feeder))
            .coerceAtMost(opportunity.pathwayCapacity)
    }.toMutableMap()

    fun renewableCapacity(opportunity: FeedingOpportunity): Double? =
        opportunity.renewableProductionRate?.let { rate ->
            state.getValue(opportunity.source.preyId).coerceAtLeast(0.0) * rate
        }

    for ((_, sourceOpportunities) in opportunities
        .filter { it.renewableProductionRate != null }
        .groupBy { it.source }) {
        val capacity = renewableCapacity(sourceOpportunities.first())!!
        val requested = sourceOpportunities.sumOf { allocated.getValue(it) }
        if (requested > capacity && requested > 0.0) {
            val scale = capacity / requested
            sourceOpportunities.forEach { opportunity ->
                allocated[opportunity] = allocated.getValue(opportunity) * scale
            }
        }
    }

    val allocationTolerance = 1e-12
    var allocationIteration = 0
    var allocationChanged = true
    while (allocationChanged && allocationIteration < opportunities.size * 2 + 8) {
        allocationIteration += 1
        val allocatedByFeeder = opportunities
            .groupBy { it.feeder }
            .mapValues { (_, feederOpportunities) ->
                feederOpportunities.sumOf { allocated.getValue(it) }
            }
        val renewableUsedBySource = opportunities
            .filter { it.renewableProductionRate != null }
            .groupBy { it.source }
            .mapValues { (_, sourceOpportunities) ->
                sourceOpportunities.sumOf { allocated.getValue(it) }
            }
        val eligible = opportunities.filter { opportunity ->
            val feederUnmet = targetIngestionByFeeder.getValue(opportunity.feeder) -
                (allocatedByFeeder[opportunity.feeder] ?: 0.0)
            val pathwayHeadroom = opportunity.pathwayCapacity -
                allocated.getValue(opportunity)
            val renewableHeadroom = renewableCapacity(opportunity)?.let { capacity ->
                capacity - (renewableUsedBySource[opportunity.source] ?: 0.0)
            } ?: Double.POSITIVE_INFINITY
            feederUnmet > allocationTolerance &&
                pathwayHeadroom > allocationTolerance &&
                renewableHeadroom > allocationTolerance
        }
        val eligibleSignalByFeeder = eligible
            .groupBy { it.feeder }
            .mapValues { (_, feederOpportunities) -> feederOpportunities.sumOf { it.signal } }
        val proposed = eligible.associateWith { opportunity ->
            val unmet = targetIngestionByFeeder.getValue(opportunity.feeder) -
                (allocatedByFeeder[opportunity.feeder] ?: 0.0)
            val weightedShare = unmet * opportunity.signal /
                eligibleSignalByFeeder.getValue(opportunity.feeder)
            minOf(
                weightedShare,
                opportunity.pathwayCapacity - allocated.getValue(opportunity),
            ).coerceAtLeast(0.0)
        }
        val renewableProposedBySource = proposed.entries
            .filter { it.key.renewableProductionRate != null }
            .groupBy { it.key.source }
            .mapValues { (_, entries) -> entries.sumOf { it.value } }

        var acceptedTotal = 0.0
        for ((opportunity, proposal) in proposed) {
            val sourceScale = renewableCapacity(opportunity)?.let { capacity ->
                val remaining = (capacity -
                    (renewableUsedBySource[opportunity.source] ?: 0.0))
                    .coerceAtLeast(0.0)
                val totalProposed = renewableProposedBySource
                    .getValue(opportunity.source)
                if (totalProposed > 0.0)
                    (remaining / totalProposed).coerceAtMost(1.0)
                else 0.0
            } ?: 1.0
            val accepted = proposal * sourceScale
            allocated[opportunity] = allocated.getValue(opportunity) + accepted
            acceptedTotal += accepted
        }
        allocationChanged = acceptedTotal > allocationTolerance
    }

    return opportunities.mapNotNull { opportunity ->
        val ingested = allocated.getValue(opportunity)
        if (ingested <= 0.0) return@mapNotNull null
        FeedingFlux(
            feeder = opportunity.feeder,
            source = opportunity.source,
            ingested = ingested,
            preyLoss = ingested * opportunity.preyLossFraction,
            assimilated = ingested * opportunity.assimilationEfficiency,
            renewableProductionRate = opportunity.renewableProductionRate,
        )
    }
}

fun carryingCapacity(
    parameters: AutotrophyParameters,
    baseCarryingCapacity: Double,
    year: Double,
    seasonsEnabled: Boolean,
): Double {
    val seasonalOffset = if (seasonsEnabled) {
        parameters.seasonalAmplitude * sin(2.0 * PI * (year - parameters.seasonalPhase))
    } else {
        0.0
    }
    return baseCarryingCapacity * (1.0 + seasonalOffset)
}

/** Extra fractional mortality per year when conditions lie outside a species' optimum. */
fun climateStressMortalityRate(
    niche: ClimateNiche,
    climate: ClimateDatumSample,
): Double {
    val temperatureDistanceC = when {
        climate.averageTemperature < niche.minOptimalTemperatureC ->
            niche.minOptimalTemperatureC - climate.averageTemperature
        climate.averageTemperature > niche.maxOptimalTemperatureC ->
            climate.averageTemperature - niche.maxOptimalTemperatureC
        else -> 0.0
    }
    val dryFraction = if (climate.precipitation < niche.minOptimalMoistureMm) {
        (niche.minOptimalMoistureMm - climate.precipitation) /
            niche.minOptimalMoistureMm.coerceAtLeast(1.0)
    } else 0.0
    val wetFraction = niche.maxOptimalMoistureMm?.let { maximum ->
        if (climate.precipitation > maximum) {
            (climate.precipitation - maximum) / maximum.coerceAtLeast(1.0)
        } else 0.0
    } ?: 0.0
    val lowLightFraction = if (climate.insolation < niche.minOptimalInsolationWm2) {
        (niche.minOptimalInsolationWm2 - climate.insolation) /
            niche.minOptimalInsolationWm2.coerceAtLeast(1.0)
    } else 0.0
    val brightLightFraction = if (climate.insolation > niche.maxOptimalInsolationWm2) {
        (climate.insolation - niche.maxOptimalInsolationWm2) /
            niche.maxOptimalInsolationWm2.coerceAtLeast(1.0)
    } else 0.0
    return niche.stressSensitivity * (
        0.06 * temperatureDistanceC +
            1.25 * dryFraction +
            0.75 * wetFraction +
            0.75 * lowLightFraction +
            0.75 * brightLightFraction
    )
}

/** Direct freeze injury not already represented by the broad climate niche. */
fun frostStressMortalityRate(
    definition: SpeciesDefinition,
    climate: ClimateDatumSample,
): Double {
    val degreesBelowFreezing = (-climate.averageTemperature).coerceAtLeast(0.0)
    val unprotectedPlantMortality = if (definition.isSessileProducer() &&
        !definition.blueprint.hasTag(EffectTag.WINTER_SURVIVAL)
    ) 0.20 * degreesBelowFreezing else 0.0
    val exposedAnimalMortality = definition.effects.filterIsInstance<ColdExposureEffect>()
        .sumOf { it.mortalityPerDegreeBelowFreezing } * degreesBelowFreezing
    return unprotectedPlantMortality + exposedAnimalMortality
}

/** Direct heat load retained by fur, blubber, and other heavy insulation. */
fun insulationHeatStressMortalityRate(
    definition: SpeciesDefinition,
    climate: ClimateDatumSample,
): Double = definition.effects.filterIsInstance<HeatRetentionEffect>().sumOf { effect ->
    (climate.averageTemperature - effect.thresholdC).coerceAtLeast(0.0) *
        effect.mortalityPerDegreeAboveThreshold
}

/** Mortality from hypoxia above the species' trait-derived optimal elevation. */
fun altitudeStressMortalityRate(definition: SpeciesDefinition, altitudeMeters: Double): Double {
    val adaptations = definition.effects.filterIsInstance<AltitudeToleranceEffect>()
    val maximumOptimalAltitude = adaptations.maxOfOrNull { it.maximumOptimalAltitudeMeters } ?: 2_500.0
    val mortalityPerKilometer = adaptations.minOfOrNull { it.mortalityPerKilometerAbove } ?: 0.45
    return ((altitudeMeters - maximumOptimalAltitude) / 1_000.0).coerceAtLeast(0.0) * mortalityPerKilometer
}

fun derivatives(year: Double, state: Biomass, model: EcosystemModel): Biomass {
    val changes = model.species.associate { it.id to 0.0 }.toMutableMap()
    val climate = model.localClimateAt(year)
    val habitatValues = habitatAxisValues(year, state, model)
    val fluxes = feedingFluxes(state, model, year, habitatValues)
    val assimilatedByFeeder = fluxes
        .groupBy { it.feeder }
        .mapValues { (_, feederFluxes) -> feederFluxes.sumOf { it.assimilated } }
    val sessileGroupBiomass = model.sessileProducerGroups.mapValues { (_, definitions) ->
        definitions.sumOf { state.getValue(it.id).coerceAtLeast(0.0) }
    }
    for (definition in model.species) {
        val biomass = state.getValue(definition.id).coerceAtLeast(0.0)
        changes[definition.id] = changes.getValue(definition.id) -
            definition.maintenanceRate * biomass
        var autotrophicEnergyGain = 0.0
        definition.autotrophy?.let { autotrophy ->
            val sessile = definition.isSessileProducer()
            val capacityDensity = if (sessile) {
                model.sessileCarryingCapacityDensity(definition.sizeClass, year)
            } else {
                carryingCapacity(
                    autotrophy,
                    autotrophy.baseCarryingCapacity,
                    year,
                    model.seasonsEnabled,
                )
            }
            val biomassUsingCapacity = if (sessile) {
                sessileGroupBiomass.getValue(definition.sizeClass)
            } else biomass
            val totalCapacity = capacityDensity * model.landArea
            val canopyLightFraction = if (sessile) {
                availableLightFraction(definition, year, state, model)
            } else 1.0
            val availableInsolation =
                (climate.insolation * canopyLightFraction).coerceAtLeast(0.0)
            val lightResponse = availableInsolation /
                (availableInsolation + autotrophy.photosynthesisHalfSaturationWm2)
            val minimumMoisture = definition.climateNiche.minOptimalMoistureMm
            val moistureResponse = if (minimumMoisture > 0.0) {
                (climate.precipitation / minimumMoisture).coerceIn(0.0, 1.0)
            } else 1.0
            val autotrophicChange = autotrophy.growthRate * lightResponse *
                moistureResponse * biomass *
                (1.0 - biomassUsingCapacity / totalCapacity)
            autotrophicEnergyGain = autotrophicChange.coerceAtLeast(0.0)
            changes[definition.id] = changes.getValue(definition.id) + autotrophicChange
        }
        definition.feeding?.let { feeding ->
            val selfDensity = biomass / model.landArea
            changes[definition.id] = changes.getValue(definition.id) -
                feeding.mortalityRate * biomass -
                feeding.densityDependence * biomass * selfDensity
            val requiredAssimilation = feeding.metabolicDemandRate * biomass
            changes[definition.id] = changes.getValue(definition.id) - requiredAssimilation
            val assimilationCoverage = if (requiredAssimilation > 0.0) {
                ((autotrophicEnergyGain +
                    (assimilatedByFeeder[definition.id] ?: 0.0)) / requiredAssimilation)
                    .coerceIn(0.0, 1.0)
            } else 1.0
            val starvationShortfall = 1.0 - assimilationCoverage
            changes[definition.id] = changes.getValue(definition.id) -
                feeding.starvationMortalityRate * starvationShortfall.pow(2.0) * biomass
        }
        val climateMortalityRate = climateStressMortalityRate(
            definition.climateNiche,
            climate,
        )
        changes[definition.id] = changes.getValue(definition.id) -
            climateMortalityRate * biomass
        val frostMortalityRate = frostStressMortalityRate(definition, climate)
        val heatRetentionMortalityRate = insulationHeatStressMortalityRate(definition, climate)
        val altitudeMortalityRate = altitudeStressMortalityRate(definition, model.altitudeMeters)
        changes[definition.id] = changes.getValue(definition.id) -
            (frostMortalityRate + heatRetentionMortalityRate + altitudeMortalityRate) * biomass
        val habitatMortalityRate = habitatStressMortalityRate(
            definition,
            habitatValues,
        )
        changes[definition.id] = changes.getValue(definition.id) -
            habitatMortalityRate * biomass
    }

    for (flux in fluxes) {
        changes[flux.prey] = changes.getValue(flux.prey) - flux.preyLoss
        changes[flux.feeder] = changes.getValue(flux.feeder) + flux.assimilated
    }

    return changes
}

## Generic effect compilation

The compiler below contains no checks for named traits. It only understands the small `TraitEffect` vocabulary defined above.

Adding a new trait usually means composing existing descriptors in the `Trait` enum. New compiler code is needed only when introducing a genuinely new category of ecological effect rather than another trait using an existing effect.

In [ ]:
fun numericValue(
    blueprint: SpeciesBlueprint,
    target: NumericTarget,
    baseValue: Double,
): Double {
    val multiplier = blueprint.effects
        .filterIsInstance<NumericMultiplier>()
        .filter { it.target == target }
        .fold(1.0) { product, effect -> product * effect.multiplier }
    val offset = blueprint.effects
        .filterIsInstance<NumericOffset>()
        .filter { it.target == target }
        .sumOf { it.offset }
    val upperBound = blueprint.effects
        .filterIsInstance<NumericUpperBound>()
        .filter { it.target == target }
        .minOfOrNull { it.maximum } ?: Double.POSITIVE_INFINITY
    return ((baseValue + offset) * multiplier).coerceAtMost(upperBound)
}

fun validateBlueprints(blueprints: List<SpeciesBlueprint>) {
    val ids = blueprints.map { it.id }
    require(ids.size == ids.toSet().size) { "Species ids must be unique" }

    for (blueprint in blueprints) {
        require(
            blueprint.hasTag(EffectTag.AUTOTROPHY) ||
                blueprint.hasTag(EffectTag.FEEDING)
        ) {
            blueprint.id + " needs an autotrophy or feeding effect"
        }

        for (requirement in blueprint.effects.filterIsInstance<RequiresTag>()) {
            require(blueprint.hasTag(requirement.tag)) {
                blueprint.id + " requires effect tag " + requirement.tag
            }
        }
        for (conflict in blueprint.effects.filterIsInstance<ConflictsWithTag>()) {
            require(!blueprint.hasTag(conflict.tag)) {
                blueprint.id + " conflicts with effect tag " + conflict.tag
            }
        }
        for (group in ExclusiveGroup.entries) {
            val count = blueprint.effects
                .filterIsInstance<ExclusiveEffect>()
                .count { it.group == group }
            require(count <= 1) {
                blueprint.id + " has multiple effects in exclusive group " + group
            }
        }
    }
}

fun deriveAutotrophy(blueprint: SpeciesBlueprint): AutotrophyParameters? {
    if (!blueprint.hasTag(EffectTag.AUTOTROPHY)) return null

    return AutotrophyParameters(
        growthRate = numericValue(
            blueprint,
            NumericTarget.AUTOTROPHY_GROWTH_RATE,
            baseValue = 1.40,
        ),
        baseCarryingCapacity = numericValue(
            blueprint,
            NumericTarget.CARRYING_CAPACITY,
            baseValue = 0.55,
        ),
        seasonalAmplitude = numericValue(
            blueprint,
            NumericTarget.SEASONAL_AMPLITUDE,
            baseValue = 0.18,
        ),
        seasonalPhase = 0.14 + 0.025 * blueprint.sizeClass.rank,
        photosynthesisHalfSaturationWm2 = numericValue(
            blueprint,
            NumericTarget.PHOTOSYNTHESIS_HALF_SATURATION_W_M2,
            baseValue = 100.0,
        ),
    )
}

fun deriveFeeding(blueprint: SpeciesBlueprint): FeedingParameters? {
    if (!blueprint.hasTag(EffectTag.FEEDING)) return null

    return FeedingParameters(
        maxConsumptionRate = numericValue(
            blueprint,
            NumericTarget.MAX_CONSUMPTION_RATE,
            baseValue = 2.50 * blueprint.sizeClass.feedingAllometry,
        ),
        assimilationEfficiency = numericValue(
            blueprint,
            NumericTarget.ASSIMILATION_EFFICIENCY,
            baseValue = 0.18,
        ),
        mortalityRate = numericValue(
            blueprint,
            NumericTarget.MORTALITY_RATE,
            baseValue = 0.06 * blueprint.sizeClass.feedingAllometry,
        ),
        metabolicDemandRate = numericValue(
            blueprint,
            NumericTarget.METABOLIC_DEMAND_RATE,
            baseValue = 0.08 * blueprint.sizeClass.feedingAllometry,
        ),
        starvationMortalityRate = numericValue(
            blueprint,
            NumericTarget.STARVATION_MORTALITY_RATE,
            baseValue = 2.20 * blueprint.sizeClass.feedingAllometry,
        ),
        densityDependence = numericValue(
            blueprint,
            NumericTarget.DENSITY_DEPENDENCE,
            baseValue = 0.90 / (1.0 + 0.18 * blueprint.sizeClass.rank),
        ),
        interference = numericValue(
            blueprint,
            NumericTarget.INTERFERENCE,
            baseValue = 0.60,
        ),
        responseExponent = numericValue(
            blueprint,
            NumericTarget.RESPONSE_EXPONENT,
            baseValue = 2.0,
        ),
    )
}

fun deriveClimateNiche(blueprint: SpeciesBlueprint): ClimateNiche {
    val bergmannShift = if (blueprint.hasTag(EffectTag.MOTILE)) {
        blueprint.sizeClass.bergmannTemperatureShiftC
    } else 0.0
    val minTemperature = numericValue(
        blueprint,
        NumericTarget.MIN_OPTIMUM_TEMPERATURE_C,
        baseValue = 5.0 + bergmannShift,
    )
    val maxTemperature = numericValue(
        blueprint,
        NumericTarget.MAX_OPTIMUM_TEMPERATURE_C,
        baseValue = 25.0 + bergmannShift,
    )
    val maximumMoisture = numericValue(
        blueprint,
        NumericTarget.MAX_OPTIMUM_MOISTURE_MM,
        baseValue = Double.POSITIVE_INFINITY,
    )
    return ClimateNiche(
        minOptimalTemperatureC = minTemperature,
        maxOptimalTemperatureC = maxTemperature,
        minOptimalMoistureMm = numericValue(
            blueprint,
            NumericTarget.MIN_OPTIMUM_MOISTURE_MM,
            baseValue = 40.0,
        ),
        maxOptimalMoistureMm = maximumMoisture.takeIf { it.isFinite() },
        minOptimalInsolationWm2 = numericValue(
            blueprint,
            NumericTarget.MIN_OPTIMUM_INSOLATION_W_M2,
            baseValue = if (blueprint.hasTag(EffectTag.MOTILE)) 40.0 else 0.0,
        ),
        maxOptimalInsolationWm2 = numericValue(
            blueprint,
            NumericTarget.MAX_OPTIMUM_INSOLATION_W_M2,
            baseValue = if (blueprint.hasTag(EffectTag.MOTILE)) 350.0 else 500.0,
        ),
        stressSensitivity = numericValue(
            blueprint,
            NumericTarget.CLIMATE_STRESS_SENSITIVITY,
            baseValue = 1.0,
        ),
    )
}

fun relativeSizePreference(
    feedingEffect: FeedsOn,
    feeder: SpeciesBlueprint,
    prey: SpeciesBlueprint,
): Double {
    val minimum = feedingEffect.minimumRelativeSize ?: return 1.0
    val maximumBonus = feeder.effects
        .filterIsInstance<RelativeSizeModifier>()
        .sumOf { it.maximumRelativeSizeBonus }
    val maximum = (feedingEffect.maximumRelativeSize ?: 0) + maximumBonus
    val gap = prey.sizeClass.rank - feeder.sizeClass.rank
    if (gap < minimum || gap > maximum) return 0.0

    return when {
        gap > 0 -> 0.80
        gap == 0 -> 1.00
        gap == -1 -> 0.85
        else -> 0.55
    }
}

fun defenseMultiplier(
    defense: DefenseEffect,
    feeder: SpeciesBlueprint,
    prey: SpeciesBlueprint,
): Double {
    val sizeAdvantage = feeder.sizeClass.rank - prey.sizeClass.rank
    if (
        defense.ignoredWhenFeederSizeAdvantageAtLeast != null &&
        sizeAdvantage >= defense.ignoredWhenFeederSizeAdvantageAtLeast
    ) {
        return 1.0
    }

    return defense.counterMultipliers
        .filterKeys { feeder.hasTag(it) }
        .values
        .maxOrNull()
        ?: defense.defaultPreferenceMultiplier
}

fun deriveDietEntries(
    feeder: SpeciesBlueprint,
    prey: SpeciesBlueprint,
): Map<FoodSource, DietEntry> {
    if (feeder.id == prey.id) return emptyMap()

    val feedingEffects = feeder.effects.filterIsInstance<FeedsOn>()
    val edibleEffects = prey.effects.filterIsInstance<EdibleAs>()

    return edibleEffects.mapNotNull { edibleEffect ->
        var preference = 0.0
        for (feedingEffect in feedingEffects) {
            if (feedingEffect.resource != edibleEffect.resource) continue
            if (
                feedingEffect.maximumPreySizeRank != null &&
                prey.sizeClass.rank > feedingEffect.maximumPreySizeRank
            ) continue
            val sizePreference = relativeSizePreference(feedingEffect, feeder, prey)
            if (sizePreference <= 0.0) continue
            val candidate = feedingEffect.preference *
                edibleEffect.preferenceMultiplier *
                sizePreference
            preference = maxOf(preference, candidate)
        }
        if (preference <= 0.0) return@mapNotNull null
        for (defense in prey.effects.filterIsInstance<DefenseEffect>()) {
            preference *= defenseMultiplier(defense, feeder, prey)
        }
        val source = FoodSource(prey.id, edibleEffect.resource)
        source to DietEntry(
            preference = preference.coerceAtLeast(0.01),
            halfSaturation = 0.06 * 1.45.pow(prey.sizeClass.rank),
            preyLossFraction = edibleEffect.preyLossFraction,
            renewableProductionRate = edibleEffect.renewableProductionRate,
        )
    }.toMap()
}

fun deriveMaintenanceRate(blueprint: SpeciesBlueprint): Double =
    0.005 + blueprint.effects
        .filterIsInstance<MaintenanceCost>()
        .sumOf { it.biomassLossRatePerYear }

fun deriveSpeciesCatalog(
    blueprints: List<SpeciesBlueprint>,
): List<SpeciesDefinition> {
    validateBlueprints(blueprints)
    val definitions = blueprints.map { blueprint ->
        val feeding = deriveFeeding(blueprint)
        val diet = if (feeding == null) {
            emptyMap()
        } else {
            blueprints.flatMap { prey ->
                deriveDietEntries(blueprint, prey).entries
            }.associate { it.key to it.value }
        }
        SpeciesDefinition(
            blueprint = blueprint,
            autotrophy = deriveAutotrophy(blueprint),
            feeding = feeding,
            climateNiche = deriveClimateNiche(blueprint),
            maintenanceRate = deriveMaintenanceRate(blueprint),
            diet = diet,
        )
    }
    validateSpeciesCatalog(definitions)
    return definitions
}

fun validateSpeciesCatalog(catalog: List<SpeciesDefinition>) {
    val ids = catalog.map { it.id }
    require(ids.size == ids.toSet().size) { "Species ids must be unique" }
    val idSet = ids.toSet()

    for (definition in catalog) {
        require(definition.maintenanceRate >= 0.0)
        definition.autotrophy?.let {
            require(it.growthRate > 0.0)
            require(it.baseCarryingCapacity > 0.0)
            require(it.seasonalAmplitude in 0.0..<1.0)
            require(it.photosynthesisHalfSaturationWm2 > 0.0)
        }
        definition.feeding?.let {
            require(it.maxConsumptionRate > 0.0)
            require(it.assimilationEfficiency in 0.0..1.0)
            require(it.mortalityRate >= 0.0)
            require(it.metabolicDemandRate > 0.0)
            require(it.starvationMortalityRate > 0.0)
            require(it.densityDependence > 0.0)
            require(it.interference >= 0.0)
            require(it.responseExponent > 1.0)
        }
        require(definition.diet.keys.all { it.preyId in idSet })
    }
}

fun validateFoodWeb(definitions: List<SpeciesDefinition>) {
    val ids = definitions.map { it.id }
    require(ids.size == ids.toSet().size) { "Species ids must be unique" }
}

In [ ]:
val speciesBlueprints = listOf(
    speciesBlueprint("grass", SizeClass.MINUSCULE, Trait.SESSILE_PRODUCER, Trait.GRASS_LIKE, Trait.LOW_FOLIAGE, Trait.COARSE_GRASS, Trait.FAST_GROWING, Trait.WAXY_CUTICLE, Trait.UNDERGROUND_MERISTEMS),
    speciesBlueprint("bamboo", SizeClass.MEDIUM, Trait.SESSILE_PRODUCER, Trait.GRASS_LIKE, Trait.LEAFY, Trait.LOW_FOLIAGE, Trait.BAMBOO_CULMS, Trait.FAST_GROWING, Trait.RHIZOMATOUS_GROWTH),
    speciesBlueprint("clover", SizeClass.MINUSCULE, Trait.SESSILE_PRODUCER, Trait.GRASS_LIKE, Trait.LEAFY, Trait.LOW_FOLIAGE, Trait.FAST_GROWING, Trait.SEED_BEARING),
    speciesBlueprint("wildflowers", SizeClass.MINUSCULE, Trait.SESSILE_PRODUCER, Trait.LEAFY, Trait.LOW_FOLIAGE, Trait.SEED_BEARING, Trait.FAST_GROWING),
    speciesBlueprint("shrub", SizeClass.MEDIUM, Trait.SESSILE_PRODUCER, Trait.LEAFY, Trait.LOW_FOLIAGE, Trait.WOODY, Trait.SLOW_GROWING, Trait.DEEP_ROOTS),
    speciesBlueprint("berry_bush", SizeClass.MEDIUM, Trait.SESSILE_PRODUCER, Trait.LEAFY, Trait.LOW_FOLIAGE, Trait.WOODY, Trait.FRUITING, Trait.SEED_BEARING, Trait.WATERLOGGING_SENSITIVE_ROOTS),
    speciesBlueprint("apple_tree", SizeClass.GIANT, Trait.SESSILE_PRODUCER, Trait.LEAFY, Trait.CANOPY_FOLIAGE, Trait.WOODY, Trait.FRUITING, Trait.SEED_BEARING, Trait.DORMANT_BUDS),
    speciesBlueprint("mango_tree", SizeClass.GIANT, Trait.SESSILE_PRODUCER, Trait.LEAFY, Trait.CANOPY_FOLIAGE, Trait.LARGE_CANOPY, Trait.WOODY, Trait.FRUITING, Trait.SEED_BEARING, Trait.DRIP_TIP_LEAVES),
    speciesBlueprint("fig_tree", SizeClass.GIANT, Trait.SESSILE_PRODUCER, Trait.LEAFY, Trait.CANOPY_FOLIAGE, Trait.LARGE_CANOPY, Trait.WOODY, Trait.FRUITING, Trait.SEED_BEARING, Trait.FAST_GROWING, Trait.DRIP_TIP_LEAVES),
    speciesBlueprint("date_palm", SizeClass.GIANT, Trait.SESSILE_PRODUCER, Trait.LEAFY, Trait.CANOPY_FOLIAGE, Trait.WOODY, Trait.FRUITING, Trait.SEED_BEARING, Trait.DEEP_ROOTS, Trait.WAXY_CUTICLE),
    speciesBlueprint("oak", SizeClass.GIANT, Trait.SESSILE_PRODUCER, Trait.LEAFY, Trait.CANOPY_FOLIAGE, Trait.LARGE_CANOPY, Trait.WOODY, Trait.SEED_BEARING, Trait.SLOW_GROWING, Trait.THICK_BARK, Trait.DORMANT_BUDS),
    speciesBlueprint("mesquite", SizeClass.MEDIUM, Trait.SESSILE_PRODUCER, Trait.LEAFY, Trait.LOW_FOLIAGE, Trait.WOODY, Trait.SEED_BEARING, Trait.DEEP_ROOTS, Trait.WAXY_CUTICLE, Trait.DROUGHT_DECIDUOUS_FOLIAGE),
    speciesBlueprint("acacia", SizeClass.GIANT, Trait.SESSILE_PRODUCER, Trait.LEAFY, Trait.CANOPY_FOLIAGE, Trait.WOODY, Trait.SEED_BEARING, Trait.DEEP_ROOTS, Trait.THICK_BARK, Trait.DROUGHT_DECIDUOUS_FOLIAGE),
    speciesBlueprint("kapok_tree", SizeClass.GIANT, Trait.SESSILE_PRODUCER, Trait.LEAFY, Trait.CANOPY_FOLIAGE, Trait.LARGE_CANOPY, Trait.WOODY, Trait.SEED_BEARING, Trait.FAST_GROWING),
    speciesBlueprint("spruce", SizeClass.GIANT, Trait.SESSILE_PRODUCER, Trait.CANOPY_FOLIAGE, Trait.LARGE_CANOPY, Trait.WOODY, Trait.SEED_BEARING, Trait.SLOW_GROWING, Trait.EVERGREEN, Trait.NEEDLE_LEAVES, Trait.THICK_BARK, Trait.DORMANT_BUDS),
    speciesBlueprint("dwarf_willow", SizeClass.MEDIUM, Trait.SESSILE_PRODUCER, Trait.LEAFY, Trait.LOW_FOLIAGE, Trait.WOODY, Trait.DEEP_ROOTS, Trait.DORMANT_BUDS, Trait.PROSTRATE_WOODY_GROWTH),
    speciesBlueprint("edelweiss", SizeClass.MINUSCULE, Trait.SESSILE_PRODUCER, Trait.LEAFY, Trait.LOW_FOLIAGE, Trait.SEED_BEARING, Trait.ALPINE_CUSHION_GROWTH),
    speciesBlueprint("cactus", SizeClass.SMALL, Trait.SESSILE_PRODUCER, Trait.LEAFY, Trait.LOW_FOLIAGE, Trait.WAXY_CUTICLE, Trait.WATER_STORAGE_TISSUE, Trait.CAM_PHOTOSYNTHESIS),
    speciesBlueprint("reeds", SizeClass.SMALL, Trait.SESSILE_PRODUCER, Trait.GRASS_LIKE, Trait.LEAFY, Trait.LOW_FOLIAGE, Trait.COARSE_GRASS, Trait.AQUATIC, Trait.FAST_GROWING),

    speciesBlueprint("sundew", SizeClass.TINY, Trait.SESSILE_PRODUCER, Trait.LEAFY, Trait.LOW_FOLIAGE, Trait.SLOW_GROWING, Trait.INSECTIVORE, Trait.AMBUSH_PREDATOR, Trait.BOG_ROOTS),
    speciesBlueprint("solar_slug", SizeClass.TINY, Trait.PHOTOSYNTHETIC, Trait.MOTILE, Trait.AQUATIC, Trait.LEAF_EATER, Trait.GROUND_FORAGING),
    speciesBlueprint("phytoplankton", SizeClass.MINUSCULE, Trait.PHOTOSYNTHETIC, Trait.MOTILE, Trait.AQUATIC),
    speciesBlueprint("krill", SizeClass.TINY, Trait.MOTILE, Trait.FILTER_FEEDER, Trait.AQUATIC, Trait.ANTIFREEZE_PROTEINS),

    speciesBlueprint("rabbit", SizeClass.SMALL, Trait.MOTILE, Trait.WARM_BLOODED, Trait.GRASS_GRAZER, Trait.LEAF_EATER, Trait.GROUND_FORAGING, Trait.BURROWING),
    speciesBlueprint("hare", SizeClass.SMALL, Trait.MOTILE, Trait.WARM_BLOODED, Trait.GRASS_GRAZER, Trait.LEAF_EATER, Trait.GROUND_FORAGING),
    speciesBlueprint("mouse", SizeClass.TINY, Trait.MOTILE, Trait.WARM_BLOODED, Trait.FRUGIVORE, Trait.SEED_EATER, Trait.BURROWING, Trait.NOCTURNAL, Trait.CONSTRICTING_PUPILS, Trait.CONCENTRATED_URINE),
    speciesBlueprint("vole", SizeClass.TINY, Trait.MOTILE, Trait.WARM_BLOODED, Trait.GRASS_GRAZER, Trait.SEED_EATER, Trait.GROUND_FORAGING, Trait.BURROWING),
    speciesBlueprint("deer", SizeClass.MEDIUM, Trait.MOTILE, Trait.WARM_BLOODED, Trait.GRASS_GRAZER, Trait.LEAF_EATER, Trait.HIGH_BROWSING, Trait.RUMINANT_STOMACH, Trait.WOODLAND_FORAGING),
    speciesBlueprint("beaver", SizeClass.SMALL, Trait.MOTILE, Trait.WARM_BLOODED, Trait.DENDROVORE, Trait.LEAF_EATER, Trait.GROUND_FORAGING, Trait.AQUATIC, Trait.BLUBBER),
    speciesBlueprint("grasshopper", SizeClass.MINUSCULE, Trait.MOTILE, Trait.GRASS_GRAZER, Trait.LEAF_EATER, Trait.GROUND_FORAGING, Trait.FLIGHT),
    speciesBlueprint("caterpillar", SizeClass.MINUSCULE, Trait.MOTILE, Trait.LEAF_EATER, Trait.GROUND_FORAGING),
    speciesBlueprint("cricket", SizeClass.MINUSCULE, Trait.MOTILE, Trait.GRASS_GRAZER, Trait.LEAF_EATER, Trait.GROUND_FORAGING, Trait.BURROWING, Trait.NOCTURNAL),
    speciesBlueprint("beetle", SizeClass.MINUSCULE, Trait.MOTILE, Trait.LEAF_EATER, Trait.SEED_EATER, Trait.GROUND_FORAGING, Trait.ARMORED),
    speciesBlueprint("fruit_fly", SizeClass.MINUSCULE, Trait.MOTILE, Trait.FRUGIVORE, Trait.FLIGHT),
    speciesBlueprint("sparrow", SizeClass.TINY, Trait.MOTILE, Trait.WARM_BLOODED, Trait.SEED_EATER, Trait.INSECTIVORE, Trait.FLIGHT, Trait.GRASSLAND_CAMOUFLAGE),
    speciesBlueprint("pigeon", SizeClass.SMALL, Trait.MOTILE, Trait.WARM_BLOODED, Trait.SEED_EATER, Trait.FRUGIVORE, Trait.FLIGHT, Trait.BRIGHT_MATING_PLUMAGE),
    speciesBlueprint("quail", SizeClass.TINY, Trait.MOTILE, Trait.WARM_BLOODED, Trait.SEED_EATER, Trait.INSECTIVORE, Trait.FLIGHT, Trait.GRASSLAND_CAMOUFLAGE),
    speciesBlueprint("boar", SizeClass.MEDIUM, Trait.MOTILE, Trait.WARM_BLOODED, Trait.OMNIVORE, Trait.GRASS_GRAZER, Trait.FRUGIVORE, Trait.GROUND_FORAGING, Trait.ARMORED, Trait.WOODLAND_FORAGING),
    speciesBlueprint("frog", SizeClass.TINY, Trait.MOTILE, Trait.INSECTIVORE, Trait.AMBUSH_PREDATOR, Trait.PROJECTILE_TONGUE, Trait.LARGE_EYES, Trait.PERMEABLE_SKIN, Trait.WETLAND_BREEDING, Trait.AQUATIC_LARVAE),
    speciesBlueprint("snake", SizeClass.SMALL, Trait.MOTILE, Trait.CARNIVORE, Trait.AMBUSH_PREDATOR, Trait.VENOMOUS),
    speciesBlueprint("weasel", SizeClass.SMALL, Trait.MOTILE, Trait.WARM_BLOODED, Trait.CARNIVORE, Trait.CHASE_PREDATOR, Trait.BURROWING),
    speciesBlueprint("fox", SizeClass.SMALL, Trait.MOTILE, Trait.WARM_BLOODED, Trait.CARNIVORE, Trait.CHASE_PREDATOR),
    speciesBlueprint("owl", SizeClass.SMALL, Trait.MOTILE, Trait.WARM_BLOODED, Trait.CARNIVORE, Trait.INSECTIVORE, Trait.AMBUSH_PREDATOR, Trait.FLIGHT, Trait.NOCTURNAL, Trait.REFLECTIVE_RETINA, Trait.CONSTRICTING_PUPILS),
    speciesBlueprint("hawk", SizeClass.SMALL, Trait.MOTILE, Trait.WARM_BLOODED, Trait.CARNIVORE, Trait.INSECTIVORE, Trait.CHASE_PREDATOR, Trait.FLIGHT, Trait.UV_FILTERING_LENSES, Trait.CONSTRICTING_PUPILS),
    speciesBlueprint("coyote", SizeClass.SMALL, Trait.MOTILE, Trait.WARM_BLOODED, Trait.CARNIVORE, Trait.PACK_HUNTER, Trait.CONCENTRATED_URINE),
    speciesBlueprint("lynx", SizeClass.SMALL, Trait.MOTILE, Trait.WARM_BLOODED, Trait.CARNIVORE, Trait.AMBUSH_PREDATOR, Trait.THICK_FUR, Trait.REFLECTIVE_RETINA),
    speciesBlueprint("wolf", SizeClass.MEDIUM, Trait.MOTILE, Trait.WARM_BLOODED, Trait.CARNIVORE, Trait.PACK_HUNTER, Trait.THICK_FUR),
    speciesBlueprint("eagle", SizeClass.SMALL, Trait.MOTILE, Trait.WARM_BLOODED, Trait.CARNIVORE, Trait.CHASE_PREDATOR, Trait.FLIGHT, Trait.UV_FILTERING_LENSES, Trait.CONSTRICTING_PUPILS),
    speciesBlueprint("bear", SizeClass.LARGE, Trait.MOTILE, Trait.WARM_BLOODED, Trait.OMNIVORE, Trait.CARNIVORE, Trait.FRUGIVORE, Trait.ARMORED, Trait.THICK_FUR),

    speciesBlueprint("dromedary", SizeClass.LARGE, Trait.MOTILE, Trait.WARM_BLOODED, Trait.GRASS_GRAZER, Trait.LEAF_EATER, Trait.HIGH_BROWSING, Trait.RUMINANT_STOMACH, Trait.HUMP_FAT, Trait.CONCENTRATED_URINE, Trait.LARGE_HEAT_RADIATING_EARS),
    speciesBlueprint("bactrian_camel", SizeClass.LARGE, Trait.MOTILE, Trait.WARM_BLOODED, Trait.GRASS_GRAZER, Trait.LEAF_EATER, Trait.HIGH_BROWSING, Trait.RUMINANT_STOMACH, Trait.HUMP_FAT, Trait.CONCENTRATED_URINE, Trait.THICK_FUR, Trait.OPEN_PLAINS_LOCOMOTION),
    speciesBlueprint("fennec_fox", SizeClass.SMALL, Trait.MOTILE, Trait.WARM_BLOODED, Trait.CARNIVORE, Trait.AMBUSH_PREDATOR, Trait.BURROWING, Trait.NOCTURNAL, Trait.LARGE_HEAT_RADIATING_EARS, Trait.SPARSE_COAT, Trait.CONCENTRATED_URINE, Trait.DESERT_CAMOUFLAGE),
    speciesBlueprint("jerboa", SizeClass.TINY, Trait.MOTILE, Trait.WARM_BLOODED, Trait.SEED_EATER, Trait.BURROWING, Trait.NOCTURNAL, Trait.LARGE_HEAT_RADIATING_EARS, Trait.CONCENTRATED_URINE),
    speciesBlueprint("oryx", SizeClass.MEDIUM, Trait.MOTILE, Trait.WARM_BLOODED, Trait.GRASS_GRAZER, Trait.LEAF_EATER, Trait.GROUND_FORAGING, Trait.RUMINANT_STOMACH, Trait.CONCENTRATED_URINE, Trait.SPARSE_COAT),
    speciesBlueprint("jaguar", SizeClass.MEDIUM, Trait.MOTILE, Trait.WARM_BLOODED, Trait.CARNIVORE, Trait.AMBUSH_PREDATOR, Trait.SPARSE_COAT, Trait.FOREST_CAMOUFLAGE),
    speciesBlueprint("sloth", SizeClass.SMALL, Trait.MOTILE, Trait.WARM_BLOODED, Trait.LEAF_EATER, Trait.HIGH_BROWSING, Trait.LOW_BASAL_METABOLISM, Trait.SPARSE_COAT),
    speciesBlueprint("toucan", SizeClass.SMALL, Trait.MOTILE, Trait.WARM_BLOODED, Trait.FRUGIVORE, Trait.FLIGHT, Trait.CONSTRICTING_PUPILS),
    speciesBlueprint("gorilla", SizeClass.LARGE, Trait.MOTILE, Trait.WARM_BLOODED, Trait.OMNIVORE, Trait.LEAF_EATER, Trait.FRUGIVORE, Trait.GROUND_FORAGING, Trait.SPARSE_COAT),
    speciesBlueprint("capybara", SizeClass.MEDIUM, Trait.MOTILE, Trait.WARM_BLOODED, Trait.GRASS_GRAZER, Trait.LEAF_EATER, Trait.GROUND_FORAGING, Trait.AQUATIC, Trait.SPARSE_COAT),
    speciesBlueprint("anaconda", SizeClass.LARGE, Trait.MOTILE, Trait.CARNIVORE, Trait.AMBUSH_PREDATOR, Trait.AQUATIC, Trait.SCALES, Trait.HEAT_SENSING_PITS),
    speciesBlueprint("tapir", SizeClass.LARGE, Trait.MOTILE, Trait.WARM_BLOODED, Trait.LEAF_EATER, Trait.FRUGIVORE, Trait.GROUND_FORAGING, Trait.SPARSE_COAT),
    speciesBlueprint("lion", SizeClass.LARGE, Trait.MOTILE, Trait.WARM_BLOODED, Trait.CARNIVORE, Trait.PACK_HUNTER, Trait.SPARSE_COAT),
    speciesBlueprint("elephant", SizeClass.GIANT, Trait.MOTILE, Trait.WARM_BLOODED, Trait.LEAF_EATER, Trait.FRUGIVORE, Trait.HIGH_BROWSING, Trait.HINDGUT_FERMENTATION, Trait.PREHENSILE_TRUNK, Trait.TUSKS, Trait.LARGE_HEAT_RADIATING_EARS),
    speciesBlueprint("giraffe", SizeClass.GIANT, Trait.MOTILE, Trait.WARM_BLOODED, Trait.LEAF_EATER, Trait.HIGH_BROWSING, Trait.RUMINANT_STOMACH, Trait.LONG_NECK, Trait.SPARSE_COAT),
    speciesBlueprint("zebra", SizeClass.MEDIUM, Trait.MOTILE, Trait.WARM_BLOODED, Trait.GRASS_GRAZER, Trait.GROUND_FORAGING, Trait.HINDGUT_FERMENTATION, Trait.SPARSE_COAT, Trait.OPEN_PLAINS_LOCOMOTION, Trait.GRASSLAND_CAMOUFLAGE),
    speciesBlueprint("hyena", SizeClass.MEDIUM, Trait.MOTILE, Trait.WARM_BLOODED, Trait.CARNIVORE, Trait.PACK_HUNTER, Trait.SPARSE_COAT),
    speciesBlueprint("wildebeest", SizeClass.LARGE, Trait.MOTILE, Trait.WARM_BLOODED, Trait.GRASS_GRAZER, Trait.GROUND_FORAGING, Trait.RUMINANT_STOMACH, Trait.SEASONAL_MIGRATION, Trait.OPEN_PLAINS_LOCOMOTION),
    speciesBlueprint("tiger", SizeClass.LARGE, Trait.MOTILE, Trait.WARM_BLOODED, Trait.CARNIVORE, Trait.AMBUSH_PREDATOR, Trait.FOREST_CAMOUFLAGE),
    speciesBlueprint("giant_panda", SizeClass.LARGE, Trait.MOTILE, Trait.WARM_BLOODED, Trait.BAMBOO_SPECIALIST, Trait.THICK_FUR, Trait.LOW_BASAL_METABOLISM),
    speciesBlueprint("crocodile", SizeClass.LARGE, Trait.MOTILE, Trait.CARNIVORE, Trait.AMBUSH_PREDATOR, Trait.AQUATIC, Trait.SCALES, Trait.ARMORED),
    speciesBlueprint("moose", SizeClass.LARGE, Trait.MOTILE, Trait.WARM_BLOODED, Trait.LEAF_EATER, Trait.GRASS_GRAZER, Trait.HIGH_BROWSING, Trait.RUMINANT_STOMACH, Trait.THICK_FUR),
    speciesBlueprint("caribou", SizeClass.MEDIUM, Trait.MOTILE, Trait.WARM_BLOODED, Trait.GRASS_GRAZER, Trait.GROUND_FORAGING, Trait.RUMINANT_STOMACH, Trait.THICK_FUR, Trait.SEASONAL_MIGRATION, Trait.OPEN_PLAINS_LOCOMOTION),
    speciesBlueprint("wolverine", SizeClass.MEDIUM, Trait.MOTILE, Trait.WARM_BLOODED, Trait.CARNIVORE, Trait.CHASE_PREDATOR, Trait.THICK_FUR),
    speciesBlueprint("ibex", SizeClass.MEDIUM, Trait.MOTILE, Trait.WARM_BLOODED, Trait.GRASS_GRAZER, Trait.LEAF_EATER, Trait.HIGH_BROWSING, Trait.RUMINANT_STOMACH, Trait.HIGH_AFFINITY_HEMOGLOBIN),
    speciesBlueprint("bison", SizeClass.LARGE, Trait.MOTILE, Trait.WARM_BLOODED, Trait.GRASS_GRAZER, Trait.GROUND_FORAGING, Trait.RUMINANT_STOMACH, Trait.THICK_FUR, Trait.OPEN_PLAINS_LOCOMOTION),
    speciesBlueprint("saiga", SizeClass.MEDIUM, Trait.MOTILE, Trait.WARM_BLOODED, Trait.GRASS_GRAZER, Trait.GROUND_FORAGING, Trait.RUMINANT_STOMACH, Trait.CONCENTRATED_URINE, Trait.SEASONAL_MIGRATION, Trait.OPEN_PLAINS_LOCOMOTION),
    speciesBlueprint("musk_ox", SizeClass.LARGE, Trait.MOTILE, Trait.WARM_BLOODED, Trait.GRASS_GRAZER, Trait.GROUND_FORAGING, Trait.RUMINANT_STOMACH, Trait.THICK_FUR, Trait.SUBCUTANEOUS_FAT),
    speciesBlueprint("arctic_fox", SizeClass.SMALL, Trait.MOTILE, Trait.WARM_BLOODED, Trait.CARNIVORE, Trait.AMBUSH_PREDATOR, Trait.THICK_FUR, Trait.REFLECTIVE_RETINA, Trait.SNOW_CAMOUFLAGE),
    speciesBlueprint("polar_bear", SizeClass.LARGE, Trait.MOTILE, Trait.WARM_BLOODED, Trait.CARNIVORE, Trait.AMBUSH_PREDATOR, Trait.THICK_FUR, Trait.BLUBBER, Trait.AQUATIC, Trait.SNOW_CAMOUFLAGE),
    speciesBlueprint("emperor_penguin", SizeClass.MEDIUM, Trait.MOTILE, Trait.WARM_BLOODED, Trait.CARNIVORE, Trait.AQUATIC, Trait.INSULATING_FEATHERS, Trait.BLUBBER, Trait.COUNTERCURRENT_HEAT_EXCHANGE),
    speciesBlueprint("monitor_lizard", SizeClass.MEDIUM, Trait.MOTILE, Trait.CARNIVORE, Trait.CHASE_PREDATOR, Trait.SCALES, Trait.HEAT_SENSING_PITS),
    speciesBlueprint("scorpion", SizeClass.TINY, Trait.MOTILE, Trait.CARNIVORE, Trait.AMBUSH_PREDATOR, Trait.VENOMOUS, Trait.NOCTURNAL, Trait.ARMORED, Trait.WAXY_EXOSKELETON, Trait.DESERT_CAMOUFLAGE),
    speciesBlueprint("kangaroo", SizeClass.MEDIUM, Trait.MOTILE, Trait.WARM_BLOODED, Trait.GRASS_GRAZER, Trait.LEAF_EATER, Trait.GROUND_FORAGING, Trait.LARGE_HEAT_RADIATING_EARS, Trait.CONCENTRATED_URINE, Trait.SPARSE_COAT, Trait.WOODLAND_FORAGING),
    speciesBlueprint("yak", SizeClass.LARGE, Trait.MOTILE, Trait.WARM_BLOODED, Trait.GRASS_GRAZER, Trait.GROUND_FORAGING, Trait.RUMINANT_STOMACH, Trait.THICK_FUR, Trait.ENLARGED_LUNGS, Trait.SUBCUTANEOUS_FAT),
    speciesBlueprint("mammoth", SizeClass.GIANT, Trait.MOTILE, Trait.WARM_BLOODED, Trait.GRASS_GRAZER, Trait.LEAF_EATER, Trait.HIGH_BROWSING, Trait.RUMINANT_STOMACH, Trait.PREHENSILE_TRUNK, Trait.TUSKS, Trait.THICK_FUR, Trait.SUBCUTANEOUS_FAT, Trait.OPEN_PLAINS_LOCOMOTION),
    speciesBlueprint("orca", SizeClass.LARGE, Trait.MOTILE, Trait.WARM_BLOODED, Trait.CARNIVORE, Trait.PACK_HUNTER, Trait.AQUATIC, Trait.BLUBBER, Trait.STREAMLINED_BODY, Trait.ECHOLOCATION),
    speciesBlueprint("harbor_seal", SizeClass.MEDIUM, Trait.MOTILE, Trait.WARM_BLOODED, Trait.CARNIVORE, Trait.AQUATIC, Trait.BLUBBER, Trait.STREAMLINED_BODY),
    speciesBlueprint("walrus", SizeClass.LARGE, Trait.MOTILE, Trait.WARM_BLOODED, Trait.BENTHIC_FORAGER, Trait.AQUATIC, Trait.BLUBBER, Trait.TUSKS),
    speciesBlueprint("great_white_shark", SizeClass.LARGE, Trait.MOTILE, Trait.CARNIVORE, Trait.CHASE_PREDATOR, Trait.AQUATIC, Trait.SCALES, Trait.STREAMLINED_BODY, Trait.ELECTRORECEPTION),
    speciesBlueprint("tuna", SizeClass.MEDIUM, Trait.MOTILE, Trait.CARNIVORE, Trait.CHASE_PREDATOR, Trait.AQUATIC, Trait.SCALES, Trait.STREAMLINED_BODY),
    speciesBlueprint("dolphin", SizeClass.LARGE, Trait.MOTILE, Trait.WARM_BLOODED, Trait.CARNIVORE, Trait.PACK_HUNTER, Trait.AQUATIC, Trait.STREAMLINED_BODY, Trait.ECHOLOCATION),
    speciesBlueprint("manta_ray", SizeClass.LARGE, Trait.MOTILE, Trait.FILTER_FEEDER, Trait.AQUATIC, Trait.STREAMLINED_BODY, Trait.ELECTRORECEPTION),
    speciesBlueprint("anglerfish", SizeClass.SMALL, Trait.MOTILE, Trait.CARNIVORE, Trait.AMBUSH_PREDATOR, Trait.AQUATIC, Trait.BIOLUMINESCENT_LURE),
    speciesBlueprint("sea_turtle", SizeClass.LARGE, Trait.MOTILE, Trait.OMNIVORE, Trait.LEAF_EATER, Trait.AQUATIC, Trait.ARMORED, Trait.SCALES, Trait.SEASONAL_MIGRATION),
    speciesBlueprint("blue_whale", SizeClass.COLOSSAL, Trait.MOTILE, Trait.WARM_BLOODED, Trait.FILTER_FEEDER, Trait.AQUATIC, Trait.BLUBBER, Trait.STREAMLINED_BODY, Trait.SEASONAL_MIGRATION),
    speciesBlueprint("whale_shark", SizeClass.COLOSSAL, Trait.MOTILE, Trait.FILTER_FEEDER, Trait.AQUATIC, Trait.SCALES, Trait.STREAMLINED_BODY),
    speciesBlueprint("albatross", SizeClass.SMALL, Trait.MOTILE, Trait.WARM_BLOODED, Trait.CARNIVORE, Trait.FLIGHT, Trait.AQUATIC, Trait.SEASONAL_MIGRATION),
)

val speciesCatalog = deriveSpeciesCatalog(speciesBlueprints)

/** Compiled species pools consumed by BiotaDistribution order assignments. */
val speciesCatalogByTaxonomicOrder: Map<TaxonomicOrder, List<SpeciesDefinition>> =
    speciesCatalog.groupBy { it.taxonomicOrder }

check(speciesBlueprints.all { it.id in taxonomicOrderBySpeciesId }) {
    "Every fixed species blueprint must have a taxonomic order"
}
check(speciesCatalogByTaxonomicOrder.values.flatten().map { it.id }.toSet() ==
    speciesCatalog.map { it.id }.toSet()
) { "The Order-to-Species mapping must cover the complete compiled catalog" }

val mesquiteClimateNiche = speciesCatalog.first { it.id == "mesquite" }.climateNiche
val cactusClimateNiche = speciesCatalog.first { it.id == "cactus" }.climateNiche
check(mesquiteClimateNiche.maxOptimalMoistureMm == 140.0) {
    "Drought-deciduous mesquite should be penalized by rainforest rainfall"
}
check(cactusClimateNiche.maxOptimalMoistureMm == 80.0) {
    "CAM cactus should be strongly waterlogging-sensitive"
}
check(Trait.LARGE_CANOPY in speciesCatalog.first { it.id == "kapok_tree" }.traits)
check(speciesCatalog.first { it.id == "frog" }.let { frog ->
    Trait.WETLAND_BREEDING in frog.traits && Trait.AQUATIC_LARVAE in frog.traits
}) { "Frogs should derive both wetland dependence and wetland refuge from traits" }

val habitatProbeSpecies = speciesCatalog.filter {
    it.id in setOf("grass", "kapok_tree", "frog", "zebra")
}
val habitatProbeModel = EcosystemModel(
    species = habitatProbeSpecies,
    seasonsEnabled = false,
    landArea = 100.0,
    climate = jungleClimate,
)
val habitatProbeState = habitatProbeSpecies.associate { definition ->
    definition.id to if (definition.id == "kapok_tree") {
        habitatProbeModel.baseSessileCarryingCapacityDensity(SizeClass.GIANT) *
            habitatProbeModel.landArea
    } else 10.0 * definition.individualBiomass
}
val habitatProbeValues = habitatAxisValues(
    0.0,
    habitatProbeState,
    habitatProbeModel,
)
check(availableLightFraction(
    speciesCatalog.first { it.id == "grass" },
    0.0,
    habitatProbeState,
    habitatProbeModel,
) <= 0.051) { "A full kapok canopy should leave about five percent ground light" }
check(habitatStressMortalityRate(
    speciesCatalog.first { it.id == "zebra" },
    habitatProbeValues,
) > 0.9) { "Closed canopy should be strongly unsuitable for open-plains locomotion" }
check(habitatRefugePreferenceMultiplier(
    speciesCatalog.first { it.id == "frog" },
    habitatProbeValues,
) <= 0.31) { "Wetland habitat should provide frogs a strong predation refuge" }

/** Representative woody producers supplied for each terrestrial sample biome. */
val representativeWoodyPlantsByBiome = mapOf(
    "desert" to setOf("mesquite"),
    "savanna" to setOf("acacia"),
    "jungle" to setOf("kapok_tree"),
    "oceanic_temperate" to setOf("oak"),
    "boreal" to setOf("spruce"),
    "tundra" to setOf("dwarf_willow"),
)
check(representativeWoodyPlantsByBiome.values.flatten().all { plantId ->
    speciesCatalog.any { it.id == plantId && Trait.WOODY in it.traits }
}) { "Every terrestrial biome example should have a woody producer" }

val frogPreyIds = speciesCatalog.first { it.id == "frog" }.diet.keys
    .filter { it.resource == FoodResource.MOTILE_PREY }
    .map { it.preyId }
    .toSet()
check(frogPreyIds.containsAll(setOf("grasshopper", "caterpillar", "cricket", "beetle", "fruit_fly"))) {
    "Frogs should have several catalog insect prey options"
}
val bambooDefinition = speciesCatalog.first { it.id == "bamboo" }
check(bambooDefinition.blueprint.hasTag(EffectTag.WINTER_SURVIVAL)) {
    "Bamboo rhizomes should protect it through temperate winters"
}
check(bambooDefinition.climateNiche.minOptimalMoistureMm >= 70.0) {
    "Bamboo should require a high-precipitation environment"
}
check(bambooDefinition.climateNiche.maxOptimalTemperatureC -
    bambooDefinition.climateNiche.minOptimalTemperatureC >= 35.0
) { "Bamboo should tolerate a broad temperature range" }
val giantPandaDiet = speciesCatalog.first { it.id == "giant_panda" }.diet.keys
check(giantPandaDiet == setOf(FoodSource("bamboo", FoodResource.BAMBOO))) {
    "Giant pandas should derive a near-exclusive bamboo diet"
}
val toucanFruitSources = speciesCatalog.first { it.id == "toucan" }.diet.keys
check(toucanFruitSources.all { it.resource == FoodResource.FRUIT } &&
    toucanFruitSources.map { it.preyId }.containsAll(
        setOf("apple_tree", "mango_tree", "fig_tree", "date_palm"),
    )
) { "Toucans should derive fruit pathways from every catalog fruit tree" }

val overbuiltChimeraBlueprint = speciesBlueprint(
    "overbuilt_chimera_test",
    TaxonomicOrder.CARNIVORA,
    SizeClass.MEDIUM, Trait.MOTILE, Trait.CARNIVORE, Trait.FLIGHT,
    Trait.BURROWING, Trait.ARMORED, Trait.VENOMOUS, Trait.REFLECTIVE_RETINA,
    Trait.CONSTRICTING_PUPILS, Trait.THICK_FUR, Trait.BLUBBER,
    Trait.COUNTERCURRENT_HEAT_EXCHANGE, Trait.CONCENTRATED_URINE,
    Trait.HUMP_FAT, Trait.TUSKS, Trait.ADAPTIVE_CAMOUFLAGE, Trait.HIBERNATION,
    Trait.SEASONAL_MIGRATION, Trait.ECHOLOCATION, Trait.ELECTRORECEPTION,
    Trait.HEAT_SENSING_PITS,
)
check(
    deriveMaintenanceRate(overbuiltChimeraBlueprint) >
        5.0 * speciesCatalog.first { it.id == "wolf" }.maintenanceRate
) { "Stacking many adaptations should create a severe energetic burden" }
check(EcosystemModel(speciesCatalog).altitudeMeters == 500.0) {
    "Notebook ecosystems should default to 500 m elevation"
}
check(speciesCatalog.first { it.id == "grass" }.blueprint.hasTag(EffectTag.WINTER_SURVIVAL)) {
    "Underground meristems should protect grass through freezing winters"
}
check(!speciesCatalog.first { it.id == "cactus" }.blueprint.hasTag(EffectTag.WINTER_SURVIVAL)) {
    "Cactus should not receive unearned winter hardiness"
}
check(speciesCatalog.first { it.id == "scorpion" }.climateNiche.minOptimalMoistureMm <
    speciesCatalog.first { it.id == "snake" }.climateNiche.minOptimalMoistureMm
) { "A waxy exoskeleton should make scorpions more drought tolerant" }
val hawkBirdPrey = speciesCatalog.first { it.id == "hawk" }.diet.keys.map { it.preyId }.toSet()
check(hawkBirdPrey.containsAll(setOf("sparrow", "pigeon", "quail"))) {
    "Flying hawks should be able to hunt the catalog's smaller birds"
}
val jerboaDefinition = speciesCatalog.first { it.id == "jerboa" }
check(Trait.BURROWING in jerboaDefinition.traits &&
    jerboaDefinition.climateNiche.maxOptimalMoistureMm == 140.0
) { "Jerboas should burrow, with tunnel flooding limiting very wet climates" }
val frogDefinition = speciesCatalog.first { it.id == "frog" }
check(frogDefinition.blueprint.hasTag(EffectTag.AERIAL_PREY_CAPTURE)) {
    "A projectile tongue should let frogs counter flying-insect evasion"
}
check(frogDefinition.diet.getValue(
    FoodSource("grasshopper", FoodResource.MOTILE_PREY),
).preference > speciesCatalog.first { it.id == "snake" }.diet.getValue(
    FoodSource("grasshopper", FoodResource.MOTILE_PREY),
).preference) { "Frogs should catch flying insects more effectively than ordinary ambush predators" }
check(setOf("bison", "caribou", "wildebeest", "bactrian_camel", "mammoth").all { id ->
    Trait.OPEN_PLAINS_LOCOMOTION in speciesCatalog.first { it.id == id }.traits
}) { "Open-country megafauna should be penalized by closed jungle canopy" }
check(insulationHeatStressMortalityRate(
    speciesCatalog.first { it.id == "mammoth" },
    ClimateDatumSample(28.0, 220.0, 250.0),
) > 0.5) { "Mammoth fur and subcutaneous fat should impose severe tropical heat stress" }
val sparseCoatedTropicalSpecies = setOf(
    "jaguar", "sloth", "gorilla", "capybara", "tapir",
    "lion", "zebra", "hyena", "kangaroo",
)
check(sparseCoatedTropicalSpecies.all { id ->
    Trait.SPARSE_COAT in speciesCatalog.first { it.id == id }.traits
}) { "Short-coated tropical mammals should use the sparse-coat trade-off" }
val rabbitFeeding = speciesCatalog.first { it.id == "rabbit" }.feeding!!
check(rabbitFeeding.mortalityRate < 0.10) {
    "Fed animals should have modest background mortality"
}
check(rabbitFeeding.starvationMortalityRate > 20.0 * rabbitFeeding.mortalityRate) {
    "Zero-food starvation should dominate background mortality"
}
val bearFeeding = speciesCatalog.first { it.id == "bear" }.feeding!!
check(abs(bearFeeding.assimilationEfficiency - 0.18) < 1e-12) {
    "Overlapping dietary niche traits must not stack global assimilation bonuses"
}
check(speciesCatalog.first { it.id == "deer" }.feeding!!.assimilationEfficiency > bearFeeding.assimilationEfficiency) {
    "Digestive physiology, not diet labels, should modify assimilation"
}
check(
    speciesCatalog.first { it.id == "dromedary" }.feeding!!.starvationMortalityRate <
        speciesCatalog.first { it.id == "oryx" }.feeding!!.starvationMortalityRate
) { "Hump fat should extend survival without food" }

/** Representative catalog animals for every Hersfeldt classification id. */
val hersfeldtIconicAnimalIds: Map<String, Set<String>> = buildMap {
    fun assign(climateIds: String, vararg animalIds: String) {
        climateIds.split(' ').forEach { climateId -> put(climateId, animalIds.toSet()) }
    }

    assign("Aha Ahh", "dromedary", "fennec_fox", "scorpion")
    assign("Ahc", "bactrian_camel", "saiga")
    assign("Ahe", "bactrian_camel", "jerboa")
    assign("Ada Adh", "oryx", "kangaroo", "fennec_fox")
    assign("Adc", "bactrian_camel", "saiga")
    assign("Ade", "oryx", "jerboa")

    assign("TUr TUrp", "jaguar", "sloth", "toucan", "gorilla")
    assign("TUf TUfp", "jaguar", "tapir", "toucan")
    assign("TUs TUsp", "elephant", "capybara", "crocodile")
    assign("TUA TUAp", "lion", "giraffe", "zebra", "wildebeest")
    assign("TQf TQfp", "tiger", "giant_panda", "tapir")
    assign("TQs TQsp", "tiger", "elephant", "capybara")
    assign("TQA TQAp", "lion", "oryx", "zebra")
    assign("TF", "jaguar", "anaconda", "owl")
    assign("TG", "monitor_lizard", "scorpion")

    assign("CTf CTfp", "tiger", "giant_panda", "tapir")
    assign("CTs CTsp", "elephant", "lion", "zebra")
    assign("CDa CDap", "deer", "fox", "bear", "beaver")
    assign("CDb CDbp", "bison", "wolf", "deer", "bear")
    assign("CEa CEap", "moose", "bear", "wolverine")
    assign("CEb CEbp", "caribou", "wolf", "moose")
    assign("CEc CEcp", "caribou", "wolverine", "musk_ox")
    assign("CMa CMb", "ibex", "boar", "fox")
    assign("CAMa CAMb", "ibex", "boar", "eagle")
    assign("CAa CAap", "bison", "saiga", "wolf")
    assign("CAb CAbp", "saiga", "bison", "wolf")
    assign("CFa", "caribou", "arctic_fox", "musk_ox")
    assign("CFb", "musk_ox", "arctic_fox", "caribou")
    assign("CG", "arctic_fox", "polar_bear")
    assign("CI", "polar_bear", "emperor_penguin")

    assign("HTf HTfp", "crocodile", "anaconda", "gorilla")
    assign("HTs HTsp", "elephant", "crocodile", "lion")
    assign("HDa HDap", "crocodile", "capybara", "monitor_lizard")
    assign("HDb HDbp", "crocodile", "anaconda", "monitor_lizard")
    assign("HDc HDcp", "monitor_lizard", "scorpion", "crocodile")
    assign("HMa HMb HMc", "ibex", "oryx", "monitor_lizard")
    assign("HAMa HAMb HAMc", "dromedary", "oryx", "ibex")
    assign("HAa HAap", "lion", "elephant", "giraffe")
    assign("HAb HAbp", "oryx", "dromedary", "scorpion")
    assign("HAc HAcp", "dromedary", "monitor_lizard", "scorpion")
    assign("HFa HFb HFc", "dromedary", "scorpion", "monitor_lizard")
    assign("HG", "scorpion", "monitor_lizard")

    assign("ETf ETfp", "mammoth", "yak", "tiger")
    assign("ETs ETsp", "bison", "mammoth", "wildebeest")
    assign("EDa EDap", "mammoth", "caribou", "wolverine")
    assign("EDb EDbp", "mammoth", "musk_ox", "arctic_fox")
    assign("EMa EMb", "ibex", "yak", "boar")
    assign("EAMa EAMb", "ibex", "saiga", "bactrian_camel")
    assign("EAa EAap", "bison", "saiga", "wildebeest")
    assign("EAb EAbp", "saiga", "yak", "mammoth")
    assign("EFa EFb", "caribou", "mammoth", "arctic_fox")
    assign("EG", "mammoth", "yak")

    assign("Ofi", "emperor_penguin", "walrus", "polar_bear", "krill")
    assign("Ofd", "orca", "harbor_seal", "walrus")
    assign("Ofg", "orca", "harbor_seal", "anglerfish")
    assign("Og", "anglerfish", "orca", "blue_whale")
    assign("Oc", "blue_whale", "orca", "harbor_seal", "tuna", "albatross", "krill")
    assign("Ot", "manta_ray", "sea_turtle", "whale_shark", "dolphin")
    assign("Oh", "sea_turtle", "manta_ray", "dolphin")
    assign("Or", "whale_shark", "manta_ray", "sea_turtle")
    assign("Oe", "blue_whale", "albatross", "orca", "sea_turtle")
}

check(hersfeldtIconicAnimalIds.size == 104) {
    "Every Hersfeldt climate classification must have iconic fauna"
}
check(hersfeldtIconicAnimalIds.values.flatten().all { animalId ->
    speciesCatalog.any { it.id == animalId }
}) { "Every iconic Hersfeldt animal must exist in the species catalog" }

/** Resolves the representative animal candidates for an actual classified tile. */
fun iconicAnimalsFor(climate: ClimateClassification): List<SpeciesDefinition> {
    val animalIds = hersfeldtIconicAnimalIds[climate.id].orEmpty()
    return speciesCatalog.filter { it.id in animalIds }
}

check(speciesCatalog.any { it.autotrophy != null && it.feeding != null }) {
    "Catalog should exercise hybrid capability composition"
}
val fruitResourceEffect = Trait.FRUITING.effects
    .filterIsInstance<EdibleAs>()
    .single { it.resource == FoodResource.FRUIT }
check(fruitResourceEffect.preyLossFraction in 0.0..<1.0)
check(fruitResourceEffect.renewableProductionRate != null)
check(speciesCatalog.first { it.id == "berry_bush" }.let { fruiter ->
    val fruitDiet = speciesCatalog.first { it.id == "mouse" }
        .dietEntry(fruiter.id, FoodResource.FRUIT)!!
    fruitDiet.preyLossFraction == fruitResourceEffect.preyLossFraction &&
        fruitDiet.renewableProductionRate == fruitResourceEffect.renewableProductionRate
}) {
    "Fruiting should create a renewable, less-destructive food pathway"
}
check(speciesCatalog.first { it.id == "grasshopper" }.let { flyingPrey ->
    val foxPreference = speciesCatalog.first { it.id == "fox" }.maximumPreferenceFor(flyingPrey.id)
    val hawkPreference = speciesCatalog.first { it.id == "hawk" }.maximumPreferenceFor(flyingPrey.id)
    hawkPreference > foxPreference
}) {
    "Flight should favor flying predators"
}

check(
    speciesCatalog.first { it.id == "owl" }.climateNiche.minOptimalInsolationWm2 <
        speciesCatalog.first { it.id == "hawk" }.climateNiche.minOptimalInsolationWm2
) { "Reflective retinas should make owls functional at lower insolation than hawks" }
check(
    speciesCatalog.first { it.id == "wolf" }.climateNiche.minOptimalTemperatureC <
        speciesCatalog.first { it.id == "coyote" }.climateNiche.minOptimalTemperatureC
) { "Thick fur should lower the wolf's optimum temperature range" }
check(
    speciesCatalog.first { it.id == "shrub" }.climateNiche.minOptimalMoistureMm <
        speciesCatalog.first { it.id == "sundew" }.climateNiche.minOptimalMoistureMm
) { "Deep roots should require less precipitation than bog roots" }

println("Trait catalog contains " + speciesCatalog.size + " compiled species")
for (definition in speciesCatalog) {
    val capabilities = listOfNotNull(
        if (definition.autotrophy != null) "autotrophy" else null,
        if (definition.feeding != null) "feeding" else null,
    ).joinToString("+")
    val feedingCosts = definition.feeding?.let { feeding ->
        " demand=%.3f/y starvation=%.2f/y".format(
            feeding.metabolicDemandRate,
            feeding.starvationMortalityRate,
        )
    }.orEmpty()
    println(
        definition.id.padEnd(14) +
            definition.sizeClass.name.padEnd(11) +
            " individual biomass=" + "%8.4f".format(definition.individualBiomass) +
            " upkeep=" + "%.3f/y".format(definition.maintenanceRate) +
            feedingCosts +
            "  " + capabilities.padEnd(18) +
            " climate=" + "%.1f..%.1f C".format(
                definition.climateNiche.minOptimalTemperatureC,
                definition.climateNiche.maxOptimalTemperatureC,
            ) +
            " / >=%.0f mm".format(definition.climateNiche.minOptimalMoistureMm) +
            " / %.0f..%.0f W/m²".format(
                definition.climateNiche.minOptimalInsolationWm2,
                definition.climateNiche.maxOptimalInsolationWm2,
            ) +
            " diet=[" + definition.diet.keys.joinToString { source ->
                source.preyId + ":" + source.resource.name.lowercase()
            } + "]"
    )
}

val fullCatalogStructure = EcosystemModel(speciesCatalog)
check(
    fullCatalogStructure.consumerNicheOverlaps.getValue("rabbit").getValue("deer") >
        fullCatalogStructure.consumerNicheOverlaps.getValue("rabbit").getValue("wolf")
) {
    "Consumers sharing plant foods should exert stronger feeding interference"
}
val rabbitDeerOverlap = fullCatalogStructure.consumerNicheOverlaps
    .getValue("rabbit")
    .getValue("deer")
check(rabbitDeerOverlap in 0.75..0.95) {
    "Ground-foraging rabbits and high-browsing deer should overlap only partially: $rabbitDeerOverlap"
}
check(fullCatalogStructure.consumerNicheOverlaps
    .getValue("rabbit")
    .getValue("bear") == 0.0) {
    "Berry foliage and berry fruit must be separate competition resources"
}
check(Trait.RUMINANT_STOMACH in speciesCatalog.first { it.id == "deer" }.traits)
val rabbitDiet = speciesCatalog.first { it.id == "rabbit" }.diet
val deerDiet = speciesCatalog.first { it.id == "deer" }.diet
check(rabbitDiet.getValue(FoodSource("berry_bush", FoodResource.LOW_FOLIAGE)).preference >
    rabbitDiet.getValue(FoodSource("oak", FoodResource.CANOPY_FOLIAGE)).preference)
check(deerDiet.getValue(FoodSource("oak", FoodResource.CANOPY_FOLIAGE)).preference >
    deerDiet.getValue(FoodSource("berry_bush", FoodResource.LOW_FOLIAGE)).preference)
val expectedMinusculeSessileGuild = speciesCatalog
    .filter { it.sizeClass == SizeClass.MINUSCULE && it.isSessileProducer() }
    .map { it.id }
    .toSet()
check(
    fullCatalogStructure.sessileProducerGroups
        .getValue(SizeClass.MINUSCULE)
        .map { it.id }
        .toSet() == expectedMinusculeSessileGuild
) {
    "Same-sized sessile producers should share one carrying-capacity guild"
}

val exampleSpeciesIds = setOf("grass", "rabbit", "wolf")
val species = speciesCatalog.filter { it.id in exampleSpeciesIds }
validateFoodWeb(species)
val ecosystem = EcosystemModel(species, seasonsEnabled = true, landArea = 1_000_000.0)

val initialBiomass: Biomass = mapOf(
    "grass" to 250_000.0,
    "rabbit" to 40_000.0,
    "wolf" to 8_000.0,
)

In [ ]:
/** Converts a climate sample to local elevation using the ecology lapse rate. */
fun climateSampleAtElevation(
    climate: ClimateDatumSample,
    elevationMeters: Double,
): ClimateDatumSample {
    require(elevationMeters in -500.0..9_000.0)
    return climate.copy(
        averageTemperature = climate.averageTemperature - 0.0065 * elevationMeters,
    )
}

/** Human-readable reasons that a species lies outside its no-stress climate envelope. */
fun speciesSuitabilityIssues(
    species: SpeciesDefinition,
    climate: ClimateDatumSample,
    elevationMeters: Double,
): List<String> {
    val local = climateSampleAtElevation(climate, elevationMeters)
    val niche = species.climateNiche
    return buildList {
        if (local.averageTemperature < niche.minOptimalTemperatureC) add("too cold")
        if (local.averageTemperature > niche.maxOptimalTemperatureC) add("too hot")
        if (local.precipitation < niche.minOptimalMoistureMm) add("too dry")
        if (niche.maxOptimalMoistureMm != null &&
            local.precipitation > niche.maxOptimalMoistureMm
        ) add("too wet")
        if (local.insolation < niche.minOptimalInsolationWm2) add("too dark")
        if (local.insolation > niche.maxOptimalInsolationWm2) add("too bright")
        if (frostStressMortalityRate(species, local) > 0.0) add("direct frost exposure")
        if (insulationHeatStressMortalityRate(species, local) > 0.0) add("insulation heat stress")
        if (altitudeStressMortalityRate(species, elevationMeters) > 0.0) add("altitude hypoxia")
    }.distinct()
}

/** True when one sample causes no climate-, frost-, insulation-, or altitude-derived stress. */
fun isSpeciesSuitedTo(
    species: SpeciesDefinition,
    climate: ClimateDatumSample,
    elevationMeters: Double,
): Boolean = speciesSuitabilityIssues(species, climate, elevationMeters).isEmpty()

val suitabilitySpotCheckElevationMeters = 500.0
val suitabilityExampleClimates = linkedMapOf(
    "oceanic_temperate" to oceanicTemperateClimate,
) + sampleClimates

println("Species suited for every month at ${suitabilitySpotCheckElevationMeters.toInt()} m:")
for ((climateName, climate) in suitabilityExampleClimates) {
    val suited = speciesCatalog.filter { species ->
        climate.months.all { sample ->
            isSpeciesSuitedTo(species, sample, suitabilitySpotCheckElevationMeters)
        }
    }
    println("%-18s %3d: %s".format(
        climateName,
        suited.size,
        suited.joinToString { it.id },
    ))
}


## Positivity-safe stochastic simulation and extinction

The deterministic dynamics use RK4 substeps. After each substep, small multiplicative environmental and species-specific shocks are applied. Noise scales with the square root of the timestep, so daily, weekly, and three-day integration have approximately the same annual variability.

A species becomes irreversibly extinct when its biomass represents fewer than two individuals (`biomass / individualBiomass < 2`). The species-specific cutoff is applied to the initial state, after disturbances, after deterministic integration, and after stochastic noise. Because zero biomass has zero growth and multiplicative noise, extinct populations cannot spontaneously reappear. Pass a seeded `Random` for reproducible experiments or `PopulationNoise.NONE` to recover deterministic behavior.

In [ ]:
data class SimulationPoint(val year: Double, val biomass: Biomass)

val minimumViableIndividuals = 2.0

fun applyIndividualExtinctionThreshold(
    state: Biomass,
    model: EcosystemModel,
    minimumIndividuals: Double = minimumViableIndividuals,
): Biomass {
    require(minimumIndividuals >= 0.0)
    return model.species.associate { definition ->
        val biomass = state.getValue(definition.id).coerceAtLeast(0.0)
        val extinctionBiomass = minimumIndividuals * definition.individualBiomass
        definition.id to if (biomass < extinctionBiomass) 0.0 else biomass
    }
}

data class PopulationNoise(
    val environmentalVolatility: Double = 0.04,
    val speciesVolatility: Double = 0.06,
) {
    init {
        require(environmentalVolatility >= 0.0)
        require(speciesVolatility >= 0.0)
    }

    companion object {
        val NONE = PopulationNoise(0.0, 0.0)
    }
}

val unitVarianceUniformLimit = sqrt(3.0)

fun Random.nextUnitVarianceNoise(): Double =
    nextDouble(-unitVarianceUniformLimit, unitVarianceUniformLimit)

fun applyPopulationNoise(
    state: Biomass,
    dt: Double,
    noise: PopulationNoise,
    random: Random,
): Biomass {
    if (noise == PopulationNoise.NONE) return state
    val stepScale = sqrt(dt)
    val environmentalShock = random.nextUnitVarianceNoise()

    return state.mapValues { (_, biomass) ->
        val speciesShock = random.nextUnitVarianceNoise()
        val proportionalChange = stepScale * (
            noise.environmentalVolatility * environmentalShock +
                noise.speciesVolatility * speciesShock
            )
        biomass * (1.0 + proportionalChange).coerceAtLeast(0.0)
    }
}

data class Disturbance(
    val year: Double,
    val remainingFraction: Map<String, Double>,
) {
    init {
        require(year >= 0.0)
        require(remainingFraction.values.all { it in 0.0..1.0 })
    }
}

fun addScaled(state: Biomass, rate: Biomass, scale: Double): Biomass =
    state.mapValues { (id, biomass) ->
        (biomass + rate.getValue(id) * scale).coerceAtLeast(0.0)
    }

fun rk4Step(year: Double, state: Biomass, dt: Double, model: EcosystemModel): Biomass {
    val k1 = derivatives(year, state, model)
    val k2 = derivatives(year + dt / 2.0, addScaled(state, k1, dt / 2.0), model)
    val k3 = derivatives(year + dt / 2.0, addScaled(state, k2, dt / 2.0), model)
    val k4 = derivatives(year + dt, addScaled(state, k3, dt), model)

    return state.mapValues { (id, biomass) ->
        val change = dt * (
            k1.getValue(id) + 2.0 * k2.getValue(id) +
                2.0 * k3.getValue(id) + k4.getValue(id)
            ) / 6.0
        (biomass + change).coerceAtLeast(0.0)
    }
}

fun simulate(
    model: EcosystemModel,
    initial: Biomass,
    years: Double,
    dt: Double = 1.0 / 365.0,
    sampleEverySteps: Int = 5,
    disturbances: List<Disturbance> = emptyList(),
    noise: PopulationNoise = PopulationNoise(),
    random: Random = Random.Default,
    minimumIndividuals: Double = minimumViableIndividuals,
): List<SimulationPoint> {
    require(dt > 0.0 && dt <= 1.0 / 52.0) { "Use weekly or smaller ecological substeps" }
    require(minimumIndividuals >= 0.0)
    require(initial.keys == model.species.map { it.id }.toSet())
    require(initial.values.all { it >= 0.0 })

    val totalSteps = (years / dt).roundToInt()
    val disturbancesByStep = disturbances.groupBy { (it.year / dt).roundToInt() }
    val result = mutableListOf<SimulationPoint>()
    var state = applyIndividualExtinctionThreshold(initial, model, minimumIndividuals)

    for (step in 0..totalSteps) {
        for (disturbance in disturbancesByStep[step].orEmpty()) {
            state = applyIndividualExtinctionThreshold(
                state.mapValues { (id, biomass) ->
                    biomass * (disturbance.remainingFraction[id] ?: 1.0)
                },
                model,
                minimumIndividuals,
            )
        }

        if (step % sampleEverySteps == 0 || step in disturbancesByStep) {
            result += SimulationPoint(step * dt, state.toMap())
        }
        if (step < totalSteps) {
            val deterministicState = applyIndividualExtinctionThreshold(
                rk4Step(step * dt, state, dt, model),
                model,
                minimumIndividuals,
            )
            state = applyIndividualExtinctionThreshold(
                applyPopulationNoise(deterministicState, dt, noise, random),
                model,
                minimumIndividuals,
            )
        }
    }

    return result
}

val starvationProbeSpecies = listOf(speciesCatalog.first { it.id == "rabbit" })
val starvationProbe = simulate(
    model = EcosystemModel(
        species = starvationProbeSpecies,
        seasonsEnabled = false,
        landArea = 1_000.0,
    ),
    initial = mapOf("rabbit" to 10.0),
    years = 3.0,
    dt = 1.0 / 365.0,
    sampleEverySteps = 365,
    noise = PopulationNoise.NONE,
)
check(starvationProbe.last().biomass.getValue("rabbit") == 0.0) {
    "A consumer with no available food should starve to extinction within three years"
}

/** Deterministic single-producer probe used to guard the sample biome envelopes. */
fun finalProducerBiomassInClimate(speciesId: String, climate: ClimateDatum): Double {
    val definition = speciesCatalog.first { it.id == speciesId }
    val probeModel = EcosystemModel(
        species = listOf(definition),
        seasonsEnabled = true,
        landArea = 10_000.0,
        climate = climate,
    )
    return simulate(
        model = probeModel,
        initial = mapOf(speciesId to 4.0 * definition.individualBiomass),
        years = 120.0,
        dt = 1.0 / 120.0,
        sampleEverySteps = 120 * 120,
        noise = PopulationNoise.NONE,
    ).last().biomass.getValue(speciesId)
}

check(finalProducerBiomassInClimate("grass", tundraClimate) > 0.0) {
    "Underground grass meristems should persist through the tundra climate"
}
check(finalProducerBiomassInClimate("dwarf_willow", tundraClimate) > 0.0) {
    "A prostrate woody producer should persist in tundra"
}
check(finalProducerBiomassInClimate("spruce", borealClimate) > 0.0) {
    "Needle-leaved evergreen trees should persist in boreal climate"
}
check(setOf("grass", "clover", "wildflowers").all { speciesId ->
    finalProducerBiomassInClimate(speciesId, desertClimate) == 0.0
}) { "Temperate meadow plants should not establish in the desert sample" }
check(finalProducerBiomassInClimate("cactus", desertClimate) > 0.0) {
    "The desert sample should support a sparse cactus population"
}
check(finalProducerBiomassInClimate("dwarf_willow", desertClimate) == 0.0) {
    "A prostrate tundra shrub should not persist in a hot desert"
}

val renewableProbeSpecies = speciesCatalog.filter {
    it.id in setOf("berry_bush", "mouse", "rabbit", "deer", "bear")
}
val renewableProbeState = mapOf(
    "berry_bush" to 1_000.0,
    "mouse" to 100.0,
    "rabbit" to 100.0,
    "deer" to 100.0,
    "bear" to 100.0,
)
val renewableProbeModel = EcosystemModel(
    renewableProbeSpecies,
    seasonsEnabled = false,
    landArea = 100.0,
)
val renewableProbeFluxes = feedingFluxes(
    renewableProbeState,
    renewableProbeModel,
)
for ((resource, productionRate) in mapOf(
    FoodResource.FRUIT to 0.30,
    FoodResource.SEED to 0.08,
)) {
    val source = FoodSource("berry_bush", resource)
    val totalIngestion = renewableProbeFluxes
        .filter { it.source == source }
        .sumOf { it.ingested }
    check(totalIngestion <= renewableProbeState.getValue(source.preyId) * productionRate + 1e-9) {
        "Consumers exceeded renewable production for $source"
    }
}
check(renewableProbeFluxes.any { flux ->
    flux.feeder == "bear" &&
        flux.source.resource == FoodResource.MOTILE_PREY &&
        flux.ingested > 0.0
}) { "A bear should redirect unmet renewable-food demand toward animal prey" }
val probeBear = renewableProbeSpecies.first { it.id == "bear" }
val probeBearFeeding = probeBear.feeding!!
val probeBearInterferenceLoad = probeBearFeeding.interference *
    renewableProbeModel.consumerNicheOverlaps.getValue("bear").entries.sumOf {
        (competitorId, overlap) ->
        overlap * renewableProbeState.getValue(competitorId)
    } / renewableProbeModel.landArea
for (flux in renewableProbeFluxes.filter {
    it.feeder == "bear" && it.source.resource == FoodResource.MOTILE_PREY
}) {
    val diet = probeBear.diet.getValue(flux.source)
    val preyDensity = renewableProbeState.getValue(flux.source.preyId) /
        renewableProbeModel.landArea
    val signal = diet.preference *
        (preyDensity / diet.halfSaturation).pow(probeBearFeeding.responseExponent)
    val pathwayCapacity = probeBearFeeding.maxConsumptionRate *
        renewableProbeState.getValue("bear") * signal /
        (1.0 + signal + probeBearInterferenceLoad)
    check(flux.ingested <= pathwayCapacity * (1.0 + 1e-10) + 1e-9) {
        "Diet switching bypassed the Type III pathway ceiling for ${flux.source}"
    }
}

val abundantHerbivoreSpecies = speciesCatalog.filter {
    it.id in setOf("grass", "oak", "berry_bush", "rabbit", "deer")
}
val abundantHerbivoreModel = EcosystemModel(
    species = abundantHerbivoreSpecies,
    seasonsEnabled = false,
    landArea = 1_000.0,
    sessileCarryingCapacityDensities = mapOf(
        SizeClass.MINUSCULE to 1_000.0,
        SizeClass.MEDIUM to 1_000.0,
        SizeClass.GIANT to 1_000.0,
    ),
)
val abundantHerbivoreHistory = simulate(
    model = abundantHerbivoreModel,
    initial = mapOf(
        "grass" to 500_000.0,
        "oak" to 500_000.0,
        "berry_bush" to 500_000.0,
        "rabbit" to 100.0,
        "deer" to 1_000.0,
    ),
    years = 120.0,
    dt = 1.0 / 120.0,
    sampleEverySteps = 120,
    noise = PopulationNoise.NONE,
)
val abundantHerbivoreFinal = abundantHerbivoreHistory.last().biomass
check(abundantHerbivoreFinal.getValue("rabbit") / abundantHerbivoreSpecies.first { it.id == "rabbit" }.individualBiomass >= minimumViableIndividuals)
check(abundantHerbivoreFinal.getValue("deer") / abundantHerbivoreSpecies.first { it.id == "deer" }.individualBiomass >= minimumViableIndividuals) {
    "Diet overlap alone must not eliminate deer while food is superabundant"
}

## Stable seasonal cycle

Start from arbitrary biomass and run long enough for the assembled ecosystem to find its own attractor. The plot normalizes each species by its observed mean over the final twenty years so differently sized species remain visible on the same scale. This reference is a measurement, not an input to the model.

In [ ]:
val seasonalHistory = simulate(
    model = ecosystem,
    initial = initialBiomass,
    years = 200.0,
)
val stableWindowStart = 180.0
val stableWindow = seasonalHistory.filter { it.year >= stableWindowStart }
val referenceBiomass: Biomass = species.associate { definition ->
    definition.id to stableWindow.map { it.biomass.getValue(definition.id) }.average()
}
val persistentReference = referenceBiomass.filterValues { it > 0.0 }
val lostSpecies = referenceBiomass.filterValues { it == 0.0 }.keys
if (lostSpecies.isNotEmpty()) println("Lost during assembly: ${lostSpecies.joinToString()}")

data class PlotPoint(
    val year: Double,
    val species: String,
    val logBiomass: Double,
)

val plotBiomassFloor = 1e-12

fun logPlotData(
    history: List<SimulationPoint>,
    speciesIds: Collection<String>,
    fromYear: Double,
): List<PlotPoint> = history
    .filter { it.year >= fromYear }
    .flatMap { point ->
        speciesIds.map { species ->
            PlotPoint(
                year = point.year,
                species = species,
                logBiomass = log10(
                    point.biomass.getValue(species).coerceAtLeast(plotBiomassFloor)
                ),
            )
        }
    }

val stableCyclePlotData = logPlotData(
    seasonalHistory,
    persistentReference.keys,
    stableWindowStart,
)

plot {
    layout.title = "Stable annual ecosystem cycle"
    layout.size = 1000 to 500
    line {
        x(stableCyclePlotData.map { it.year }, name = "year")
        y(stableCyclePlotData.map { it.logBiomass }, name = "log10 biomass")
        color(stableCyclePlotData.map { it.species }, name = "species")
    }
}

In [ ]:
data class StabilityMetrics(
    val species: String,
    val mean: Double,
    val minimum: Double,
    val maximum: Double,
    val coefficientOfVariation: Double,
    val minimumToMean: Double,
)

fun stabilityMetrics(
    history: List<SimulationPoint>,
    speciesIds: Collection<String>,
    fromYear: Double,
): List<StabilityMetrics> {
    val window = history.filter { it.year >= fromYear }
    require(window.isNotEmpty())

    return speciesIds.map { species ->
        val values = window.map { it.biomass.getValue(species) }
        val mean = values.average()
        val standardDeviation = sqrt(values.sumOf { (it - mean).pow(2) } / values.size)
        StabilityMetrics(
            species = species,
            mean = mean,
            minimum = values.minOrNull()!!,
            maximum = values.maxOrNull()!!,
            coefficientOfVariation = if (mean > 0.0) standardDeviation / mean else 0.0,
            minimumToMean = if (mean > 0.0) values.minOrNull()!! / mean else 0.0,
        )
    }
}

val stableMetrics = stabilityMetrics(
    history = seasonalHistory,
    speciesIds = referenceBiomass.keys,
    fromYear = stableWindowStart,
)

println("species    mean      min       max       CV       min/mean")
stableMetrics.forEach { metric ->
    println(
        "%-8s  %.4f    %.4f    %.4f    %.3f    %.3f".format(
            metric.species, metric.mean, metric.minimum, metric.maximum,
            metric.coefficientOfVariation, metric.minimumToMean,
        )
    )
}

## Recovery from a major disruption

After allowing the annual cycle to settle, remove 65% of the rabbits at year 10. No population floor or immigration rescues them: recovery comes entirely from the local growth and feeding equations.

In [ ]:
val disturbanceYear = 10.0
val settledState = seasonalHistory.last().biomass
val disruptedHistory = simulate(
    model = ecosystem,
    initial = settledState,
    years = 50.0,
    disturbances = listOf(
        Disturbance(
            year = disturbanceYear,
            remainingFraction = mapOf("rabbit" to 0.35),
        )
    ),
)

val disruptionPlotData = logPlotData(disruptedHistory, persistentReference.keys, fromYear = 5.0)

plot {
    layout.title = "Recovery after removing 65% of rabbits at year 10"
    layout.size = 1000 to 500
    line {
        x(disruptionPlotData.map { it.year }, name = "year")
        y(disruptionPlotData.map { it.logBiomass }, name = "log10 biomass")
        color(disruptionPlotData.map { it.species }, name = "species")
    }
}

In [ ]:
fun sustainedRecoveryTime(
    history: List<SimulationPoint>,
    species: String,
    target: Double,
    afterYear: Double,
    tolerance: Double = 0.10,
    holdYears: Double = 2.0,
): Double? {
    val after = history.filter { it.year >= afterYear }
    for ((index, candidate) in after.withIndex()) {
        val endYear = candidate.year + holdYears
        val holdWindow = after.subList(index, after.size).takeWhile { it.year <= endYear }
        val spansHoldPeriod = holdWindow.lastOrNull()?.year?.let { it >= endYear - 0.02 } == true
        val staysInBand = holdWindow.all {
            abs(it.biomass.getValue(species) / target - 1.0) <= tolerance
        }
        if (spansHoldPeriod && staysInBand) return candidate.year - afterYear
    }
    return null
}

println("species    minimum after shock    sustained recovery to +/-10%")
for ((species, target) in persistentReference) {
    val afterShock = disruptedHistory.filter { it.year >= disturbanceYear }
    val minimumFraction = afterShock.minOf { it.biomass.getValue(species) / target }
    val recovery = sustainedRecoveryTime(
        disruptedHistory, species, target, afterYear = disturbanceYear
    )
    val recoveryText = recovery?.let { "%.1f years".format(it) } ?: "not within simulation"
    println("%-8s   %.3f                  %s".format(species, minimumFraction, recoveryText))
}

## Random ecosystem assemblies

Each trial samples already compiled species without changing their traits or derived parameters. One autotrophic species is guaranteed as an energy source; all remaining slots are sampled blindly. Diet entries may point to absent prey, feeding species may starve, and sampled species may exclude one another.

The test uses a three-day integration step, checks every biomass for finite nonnegative values, and calls a species persistent when its final twenty-year mean exceeds the reporting threshold.

In [ ]:
fun randomSpeciesSample(
    catalog: List<SpeciesDefinition>,
    count: Int,
    random: Random,
): List<SpeciesDefinition> {
    require(count in 1..catalog.size)
    val autotroph = catalog.filter { it.autotrophy != null }.random(random)
    return listOf(autotroph) + (catalog - autotroph).shuffled(random).take(count - 1)
}

fun randomInitialBiomass(
    model: EcosystemModel,
    random: Random,
): Biomass = model.species.associate { definition ->
    val initialDensity = when {
        definition.isSessileProducer() -> {
            val guildSize = model.sessileProducerGroups
                .getValue(definition.sizeClass)
                .size
            model.sessileCarryingCapacityDensity(definition.sizeClass, year = 0.0) *
                random.nextDouble(0.10, 0.65) / guildSize
        }
        definition.autotrophy != null ->
            definition.autotrophy.baseCarryingCapacity *
                random.nextDouble(0.10, 0.65)
        else -> random.nextDouble(0.002, 0.060)
    }
    val densityBasedBiomass = initialDensity * model.landArea
    val foundingBiomass = 4.0 * definition.individualBiomass
    definition.id to maxOf(densityBasedBiomass, foundingBiomass)
}

data class RandomAssemblyResult(
    val seed: Int,
    val selected: List<String>,
    val persistent: List<String>,
    val lost: List<String>,
    val maximumPersistentCv: Double,
)

fun runRandomAssembly(
    catalog: List<SpeciesDefinition>,
    speciesCount: Int,
    seed: Int,
    years: Double = 120.0,
    measurementYears: Double = 20.0,
): RandomAssemblyResult {
    val random = Random(seed)
    val selected = randomSpeciesSample(catalog, speciesCount, random)
    validateFoodWeb(selected)
    val model = EcosystemModel(selected, seasonsEnabled = true, landArea = 100_000.0)
    val history = simulate(
        model,
        randomInitialBiomass(model, random),
        years = years,
        dt = 1.0 / 120.0,
        sampleEverySteps = 4,
        random = random,
    )
    check(history.all { point ->
        point.biomass.values.all { it.isFinite() && it >= 0.0 }
    }) { "Assembly " + seed + " produced invalid biomass" }

    val metrics = stabilityMetrics(
        history,
        selected.map { it.id },
        fromYear = years - measurementYears,
    )
    val persistent = metrics.filter { it.mean > 0.0 }
    val persistentIds = persistent.map { it.species }.sorted()
    val selectedIds = selected.map { it.id }.sorted()
    return RandomAssemblyResult(
        seed,
        selectedIds,
        persistentIds,
        selectedIds - persistentIds.toSet(),
        persistent.maxOfOrNull { it.coefficientOfVariation } ?: 0.0,
    )
}

val randomAssemblyResults = (0..<12).map { index ->
    runRandomAssembly(speciesCatalog, speciesCount = 10, seed = 10_000 + index)
}

println("seed    selected  persistent  lost  max persistent CV")
for (result in randomAssemblyResults) {
    println("%-7d %-9d %-11d %-5d %.3f".format(
        result.seed,
        result.selected.size,
        result.persistent.size,
        result.lost.size,
        result.maximumPersistentCv,
    ))
    println("  selected:   " + result.selected.joinToString())
    println("  persistent: " + result.persistent.joinToString())
    if (result.lost.isNotEmpty()) println("  lost:       " + result.lost.joinToString())
}

check(randomAssemblyResults.all { result ->
    result.persistent.isNotEmpty() && result.selected.any { id ->
        speciesCatalog.first { it.id == id }.autotrophy != null
    }
})

In [ ]:
// Change this seed to draw another reproducible five-species ecosystem.
val randomFiveSeed = Random.nextInt()
val randomFiveRandom = Random(randomFiveSeed)
val randomFiveSpecies = randomSpeciesSample(
    catalog = speciesCatalog,
    count = 5,
    random = randomFiveRandom,
)
val randomFiveModel = EcosystemModel(randomFiveSpecies, seasonsEnabled = true, landArea = 100_000.0)
val randomFiveInitial = randomInitialBiomass(randomFiveModel, randomFiveRandom)
val randomFiveHistory = simulate(
    model = randomFiveModel,
    initial = randomFiveInitial,
    years = 120.0,
    dt = 1.0 / 120.0,
    sampleEverySteps = 4,
    random = randomFiveRandom,
)

println("Random five-species sample: " + randomFiveSpecies.joinToString { it.id })

val randomFivePlotData = logPlotData(
    randomFiveHistory,
    randomFiveSpecies.map { it.id },
    fromYear = 0.0,
)

plot {
    layout.title = "Random five-species ecosystem (seed " + randomFiveSeed + ")"
    layout.size = 1000 to 500
    line {
        x(randomFivePlotData.map { it.year }, name = "year")
        y(randomFivePlotData.map { it.logBiomass }, name = "log10 biomass")
        color(randomFivePlotData.map { it.species }, name = "species")
    }
}

In [ ]:
//val selectedSpecies = speciesCatalog.filter { it.id in listOf("grass", "oak", "berry_bush", "deer", "rabbit", "vole", "fox", "wolf", "ibex", "horse", "zebra", "bear", "lion", "tiger", "giant_panda", "bamboo" ) }
val selectedSpecies = speciesCatalog.filter { (Trait.AQUATIC !in it.traits && Random.nextDouble() < 0.5) || it.id in listOf("grass") }
//val selectedSpecies = speciesCatalog
val selectedSessileCarryingCapacityDensities = mapOf(
    SizeClass.MINUSCULE to 500.00,
    SizeClass.GIANT to 100.80,
    SizeClass.MEDIUM to 100.0,
)
val selectedClimate =
    listOf(oceanicTemperateClimate to "oceanic temperate", desertClimate to "desert", savannaClimate to "savanna", jungleClimate to "jungle", borealClimate to "boreal", tundraClimate to "tundra", iceCapClimate to "ice cap")
    .filter { it.second != "ice cap" }
    .random()
val selectedLandArea = 100000.0
val selectedModel = EcosystemModel(
    species = selectedSpecies,
    seasonsEnabled = true,
    landArea = selectedLandArea,
    sessileCarryingCapacityDensities = selectedSessileCarryingCapacityDensities,
    climate = selectedClimate.first
)
val selectedInitial = randomInitialBiomass(selectedModel, Random)
val selectedHistory = simulate(
    model = selectedModel,
    initial = selectedInitial,
    years = 200.0,
    dt = 1.0 / 120.0,
    sampleEverySteps = 4,
    noise = PopulationNoise(
        environmentalVolatility = 0.2,
        speciesVolatility = 0.3,
    ),
)

println("Selected climate: ${selectedClimate.second}")
println("Selected-species sample: " + selectedSpecies.joinToString { it.id })

println("Survived: ${selectedHistory.last().biomass.filter { it.value > 0 }.map { it.key }}")
println("Extinct: ${selectedHistory.last().biomass.filter { it.value == 0.0 }.map { it.key }}")

val selectedPlotData = logPlotData(
    selectedHistory,
    selectedSpecies.map { it.id },
    fromYear = 0.0,
)

plot {
    layout.title = "Custom ecosystem"
    layout.size = 1000 to 500
    line {
        x(selectedPlotData.map { it.year }, name = "year")
        y(selectedPlotData.map { it.logBiomass }, name = "log10 biomass")
        color(selectedPlotData.map { it.species }, name = "species")
    }
}

## Useful experiments

- Change the sample count, sample size, or seeds in randomAssemblyResults.
- Change seasonalAmplitude to control the environmental cycle strength.
- Move responseExponent toward 1.0 to compare Type III feeding with Type II feeding.
- Reduce densityDependence to weaken disease, territoriality, and other self-crowding losses.
- Reduce interference to weaken intake obstruction from consumers with overlapping diets.
- Inspect consumerNicheOverlaps to see which feeding guilds interfere most strongly when food is scarce.
- Change FRUITING or SEED_BEARING renewableProductionRate values to alter annual renewable-food supply.
- Compare GROUND_FORAGING, HIGH_BROWSING, and RUMINANT_STOMACH against plants exposing LOW_FOLIAGE, CANOPY_FOLIAGE, and COARSE_GRASS.
- Change sessileCarryingCapacityDensities to alter the shared producer capacity of each size class.
- Add trait-only species blueprints and inspect their compiled parameters and diets.
- Stack concrete adaptations such as THICK_FUR, BLUBBER, DEEP_ROOTS, WAXY_CUTICLE, or CONCENTRATED_URINE and inspect the resulting climate niche.
- Replace oceanicTemperateClimate with any tile ClimateDatum from ClimateSimulation.
- Use sampleClimates["desert"], ["savanna"], ["jungle"], ["boreal"], ["tundra"], or ["ice_cap"] for reference-biome experiments.
- Compare combinations of REFLECTIVE_RETINA, LARGE_EYES, CONSTRICTING_PUPILS, and UV_FILTERING_LENSES across insolation regimes.
- Sample the entire catalog to study competition, omnivory, and trophic cascades.
- Use hersfeldtIconicAnimalIds[climateId] as a climate-specific candidate pool; these are alternatives associated with the zone, not a claim that every listed animal coexists.

Extinction occurs when a species has fewer than `minimumViableIndividuals` (two by default), computed from total biomass and the species' size-derived `individualBiomass`. Pass a different `minimumIndividuals` to `simulate` to test another demographic cutoff.